In [1]:
#Imports the things

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import scipy as sp
from datetime import datetime, date, time, timedelta
import gurobipy
import copy

In [3]:
#Define the Locations and transportation methods

locations = {}

methods = {}

original_methods = [
    "ped_public",
    "bicycle",
    "motorcycle"
]

original_places = [
    
    #Boulder Locations
    "Central Pearl",
    "Springdale",
    "CU Engineering",
    "CU Math",
    "CU Duane",
    "CU IMIG",
    "East Pearl",
    "Boulder Hill",
    "Baseline & 30th",
    "Valmont",
    "North 28th",

    #VG Locations
    "VG",

    #Estes Locations
    "Estes",

    #Denver Locations
    "Union Station",
    "East Colfax",
    "Auraria",
    "Coors Field",
    "DIA",

    #Misc other
    "Interlocken",
    "Euro Moto Electric"
]

for i, x in enumerate(original_methods):
    methods[x] = i

for i, x in enumerate(original_places):
    locations[x] = i

inv_locations = {v: k for k, v in locations.items()}

#Define the travel time values matrix

travel_time = np.full((len(locations), len(locations), len(methods)), np.timedelta64(timedelta(minutes=100000000)))

for i in range(len(travel_time[:,0,0])):
    travel_time[i, i, methods["ped_public"]] = np.timedelta64(timedelta(minutes=0))
#travel times for pedestrian/public transit
travel_time_ped = [
    #Hub Connections
    ["Central Pearl", "Union Station", timedelta(minutes=60)],
    ["Union Station", "VG", timedelta(minutes=40)],
    ["Union Station", "Estes", timedelta(minutes=40)],

    #Boulder Connections
    ["Central Pearl", "Springdale", timedelta(minutes=20)],
    ["Central Pearl", "CU Engineering", timedelta(minutes=15)],
    ["Central Pearl", "East Pearl", timedelta(minutes=10)],

    ["Springdale", "CU Engineering", timedelta(minutes=10)],
    ["Springdale", "CU Math", timedelta(minutes=14)],
    ["Springdale", "CU Duane", timedelta(minutes=15)],
    ["Springdale", "CU IMIG", timedelta(minutes=20)],
    ["Springdale", "East Pearl", timedelta(minutes=15)],
    ["Springdale", "Boulder Hill", timedelta(minutes=30)],
    ["Springdale", "Baseline & 30th", timedelta(minutes=15)],
    ["Springdale", "Valmont", timedelta(minutes=45)],

    ["CU Engineering", "CU Math", timedelta(minutes=5)],
    ["CU Engineering", "CU IMIG", timedelta(minutes=10)],
    ["CU Engineering", "Baseline & 30th", timedelta(minutes=15)],
    
    ["CU Math", "CU Duane", timedelta(minutes=5)],
    ["CU Duane", "CU IMIG", timedelta(minutes=8)],

    ["CU IMIG", "Boulder Hill", timedelta(minutes=15)],

    ["North 28th", "Springdale", timedelta(minutes=15)],
    ["North 28th", "CU Engineering", timedelta(minutes=20)],
    ["North 28th", "East Pearl", timedelta(minutes=15)],


    #Denver Connections
    ["VG", "Estes", timedelta(minutes=60)],
    ["VG", "Union Station", timedelta(minutes=60)],

    ["Estes", "Union Station", timedelta(minutes=50)],
    
    ["Union Station", "East Colfax", timedelta(minutes=25)],
    ["Union Station", "Auraria", timedelta(minutes=15)],
    ["Union Station", "Coors Field", timedelta(minutes=10)],

    #DIA Connections
    ["Springdale", "DIA", timedelta(minutes=75)],
    ["VG", "DIA", timedelta(minutes=120)],

    ["Interlocken", "Union Station", timedelta(minutes=30)],
    ["Interlocken", "Central Pearl", timedelta(minutes=30)]
    
]

travel_time_bike = [
    
    #Boulder Network
    ["Central Pearl", "Springdale", timedelta(minutes=10)],
    ["Central Pearl", "CU Engineering", timedelta(minutes=10)],
    ["Central Pearl", "CU IMIG", timedelta(minutes=14)],
    ["Central Pearl", "East Pearl", timedelta(minutes=2)],
    ["Central Pearl", "Boulder Hill", timedelta(minutes=5)],
    
    ["Springdale", "CU Engineering", timedelta(minutes=5)],
    ["Springdale", "CU Math", timedelta(minutes=6)],
    ["Springdale", "CU IMIG", timedelta(minutes=8)],
    ["Springdale", "East Pearl", timedelta(minutes=9)],
    ["Springdale", "Baseline & 30th", timedelta(minutes=8)],
    ["Springdale", "Valmont", timedelta(minutes=20)],
    
    ["CU Engineering", "CU Math", timedelta(minutes=1)],
    ["CU Math", "CU Duane", timedelta(minutes=2)],
    ["CU Engineering", "CU IMIG", timedelta(minutes=5)],
    ["CU Duane", "CU IMIG", timedelta(minutes=2)],
    ["CU IMIG", "Boulder Hill", timedelta(minutes=5)],

    ["North 28th", "Springdale", timedelta(minutes=5)],
    ["North 28th", "CU Engineering", timedelta(minutes=10)],
    ["North 28th", "East Pearl", timedelta(minutes=5)],

    #VG/Estes/Denver Network
    ["VG", "Estes", timedelta(minutes=20)],
    ["Estes", "Union Station", timedelta(minutes=45)],
    ["Union Station", "Auraria", timedelta(minutes=5)],
    ["Union Station", "East Colfax", timedelta(minutes=15)],
    ["Union Station", "Coors Field", timedelta(minutes=5)],

    #Interlocken
    ["Interlocken", "Estes", timedelta(minutes=45)],
    ["Interlocken", "Springdale", timedelta(minutes=45)]
]

travel_time_motorcycle = [

    #Long-distance links only
    ["Springdale", "VG", timedelta(minutes=50)],
    ["VG", "Estes", timedelta(minutes=10)],
    ["VG", "Union Station", timedelta(minutes=35)],
    ["Estes", "Union Station", timedelta(minutes=30)],
    ["Union Station", "East Colfax", timedelta(minutes=15)],
    ["VG", "East Colfax", timedelta(minutes=45)],
    ["Estes", "East Colfax", timedelta(minutes=45)],

    ["Interlocken", "Springdale", timedelta(minutes=25)],
    ["Interlocken", "VG", timedelta(minutes=35)],
    ["Interlocken", "Estes", timedelta(minutes=30)],

    ["Euro Moto Electric", "Springdale", timedelta(minutes=45)],
    ["Euro Moto Electric", "VG", timedelta(minutes=30)],
    ["Euro Moto Electric", "Estes", timedelta(minutes=20)],
    ["Euro Moto Electric", "Union Station", timedelta(minutes=15)]
]
#Function to make sure location is good
def check_location(locations:dict, name:str):
    if name in locations:
        return True
    else:
        return False
#Functino populates the travel matrix!
for i, link in enumerate(travel_time_ped):
    
    #Makes sure names are correct
    if not check_location(locations, link[0]):
        print(f"Failure! : {link[0]}")
    elif not check_location(locations, link[1]):
        print(f"Failure! : {link[1]}")
    
    else:
        name1 = link[0]
        name2 = link[1]
    
    travel_time[locations[name1], locations[name2], methods["ped_public"]] = np.timedelta64(link[2])
    travel_time[locations[name2], locations[name1], methods["ped_public"]] = np.timedelta64(link[2])

for i, link in enumerate(travel_time_bike):
    
    #Makes sure names are correct
    if not check_location(locations, link[0]):
        print(f"Failure! : {link[0]}")
    elif not check_location(locations, link[1]):
        print(f"Failure! : {link[1]}")

    else:
        name1 = link[0]
        name2 = link[1]
    
    travel_time[locations[name1], locations[name2], methods["bicycle"]] = np.timedelta64(link[2])
    travel_time[locations[name2], locations[name1], methods["bicycle"]] = np.timedelta64(link[2])

for i, link in enumerate(travel_time_motorcycle):
    
    #Makes sure names are correct
    if not check_location(locations, link[0]):
        print(f"Failure! : {link[0]}")
    elif not check_location(locations, link[1]):
        print(f"Failure! : {link[1]}")

    else:
        name1 = link[0]
        name2 = link[1]
    
    travel_time[locations[name1], locations[name2], methods["motorcycle"]] = np.timedelta64(link[2])
    travel_time[locations[name2], locations[name1], methods["motorcycle"]] = np.timedelta64(link[2])
bike_location = "Springdale"
motorcycle_location = "VG"
#Quick thing to check if a node has been visited or not
def is_visited(visited:list, j:int):
    for k in range(len(visited)):
        if visited[k][0] == j:
            return True
    return False

#Find the source from the dijkstra
def source(visited:list, j:int):
    if not is_visited(visited, j):
        return j
    else:
        for k in range(len(visited)):
            if visited[k][0] == j:
                return visited[k][1]


#The function for a travel time mode!
def travel_time_mode(travel_time, start, end, method):

    grid = (travel_time[:,:,methods[method]].copy()).astype('int') / (60*10**6)

    visited = [
        [locations[start], locations[start], 0]
    ]

    while not is_visited(visited, locations[end]):

        travel_val_min = np.nan
        tvm_start = np.nan
        tvm_end = np.nan  

        for i in range(len(visited)): 

            for j in range(len(grid[i, :])):
                #print(f"Cheking at {visited[i][0]}, {j}:")


                travel_val = visited[i][2] + grid[visited[i][0], j]

                #print(f"Travel val is {travel_val}")

                if np.isnan(travel_val_min) and not is_visited(visited,j):
                    travel_val_min = travel_val
                    tvm_start = i
                    tvm_end = j
                elif travel_val < travel_val_min and not is_visited(visited,j):
                    travel_val_min = travel_val
                    tvm_start = visited[i][0]
                    tvm_end = j 

        visited.append([tvm_end, tvm_start, travel_val_min])

        #print([tvm_end, tvm_start, travel_val_min])

    if visited[-1][2] >10**6:
        return False

    backtrack = [locations[end]]
    while backtrack[-1] != locations[start]:
        backtrack.append(source(visited, backtrack[-1]))

    backtrack.reverse()

    return [visited[-1][2], backtrack]
#Find the travel time function!
def travel_time_all(travel_time, start, end):

    grid = (travel_time.copy()).astype('int') / (60*10**6)

    visited = [
        [locations[start], locations[start], 0]
    ]

    has_bike = False
    has_motorcycle = False

    if start == bike_location:
        has_bike = True
    if start == motorcycle_location:
        has_motorcycle = True

    while not is_visited(visited, locations[end]):

        travel_val_min = np.nan
        tvm_start = np.nan
        tvm_end = np.nan  

        #Check places accessible via ped
        for i in range(len(visited)): 

            for j in range(len(grid[i, :])):
                #print(f"Cheking at {visited[i][0]}, {j}:")


                travel_val = visited[i][2] + grid[visited[i][0], j, methods["ped_public"]]

                #print(f"Travel val is {travel_val}")

                if np.isnan(travel_val_min) and not is_visited(visited,j):
                    travel_val_min = travel_val
                    tvm_start = i
                    tvm_end = j
                elif travel_val < travel_val_min and not is_visited(visited,j):
                    travel_val_min = travel_val
                    tvm_start = visited[i][0]
                    tvm_end = j 

        #Check places accessible via bicycle
        if has_bike:
            for i in range(len(visited)): 

                for j in range(len(grid[i, :])):
                    #print(f"Cheking at {visited[i][0]}, {j}:")


                    travel_val = visited[i][2] + grid[visited[i][0],j, [methods["bicycle"]]]

                    #print(f"Travel val is {travel_val}")

                    if np.isnan(travel_val_min) and not is_visited(visited,j):
                        travel_val_min = travel_val
                        tvm_start = i
                        tvm_end = j
                    elif travel_val < travel_val_min and not is_visited(visited,j):
                        travel_val_min = travel_val
                        tvm_start = visited[i][0]
                        tvm_end = j 

        if has_motorcycle:
            for i in range(len(visited)): 

                for j in range(len(grid[i, :])):
                    #print(f"Cheking at {visited[i][0]}, {j}:")


                    travel_val = visited[i][2] + grid[visited[i][0],j,[methods["motorcycle"]]]

                    #print(f"Travel val is {travel_val}")

                    if np.isnan(travel_val_min) and not is_visited(visited,j):
                        travel_val_min = travel_val
                        tvm_start = i
                        tvm_end = j
                    elif travel_val < travel_val_min and not is_visited(visited,j):
                        travel_val_min = travel_val
                        tvm_start = visited[i][0]
                        tvm_end = j 

        visited.append([tvm_end, tvm_start, travel_val_min])
        if tvm_end == bike_location:
            has_bike = True
        if tvm_end == motorcycle_location:
            has_motorcycle = True

        #print([tvm_end, tvm_start, travel_val_min])

    if visited[-1][2] >10**6:
        return False

    backtrack = [locations[end]]
    while backtrack[-1] != locations[start]:
        backtrack.append(source(visited, backtrack[-1]))

    backtrack.reverse()

    return [visited[-1][2], backtrack]

bike_location = "Springdale"
motorcycle_location = "Springdale"

In [3]:
#Helper functions, not class specific

def get_weekday_str(day_date:date):
    """
    Returns the weekday as a string for a given date.
    """
    return day_date.strftime("%A")[:3]

print(get_weekday_str(date(2023, 10, 1)))  # Example usage

Sun


In [4]:
#Define the time mode class and initializes the modes

class Time_mode:

  def __init__(self):

    self.daily_tasks = []
    self.weekly_tasks = []
    self.events = []
    self.tasks = {}
    
  def assign_name(self, name):
    if not isinstance(name, str):
      raise TypeError("Wrong datatype to be calling time mode")
    
    self.name = name
    return
  
  def clear_daily_tasks(self):
      self.daily_tasks = []

  def clear_events(self, name:str=None, v:bool=False):
    if name is None:
      self.events = []
    else:
      print(f"Trying to find {name} in {self.name} events")
      for i in range(len(self.events)):
        if self.events[i].name == name:
          print(f"Found {name} in {self.name} events")
          del self.events[i]
          return
      else:
        if v: print(f"Could not find {name} in {self.name} events")

  def clear_tasks(self):
    self.tasks = {}

  def print_Daily_tasks(self):
    for i, x in enumerate(self.Daily_tasks):
      x.print_goal_info()
      print("\n")

  def print_events(self):
    for i, x in enumerate(self.events):
      x.print_event_info()
      print("\n")

  def print_long_term_goals(self):
    for i, x in enumerate(self.long_term_goals):
      x.print_goal_info()
      print("\n")

  def clear_weekly_tasks(self):
    self.weekly_tasks = []


mode_ids = [
    'rd',   #read
    'vs',   #videogames & shows
    'hb',   #hobbies
    'mh',   #mental health 
    'ph',   #physical health
    'hg',   #hygiene
    'ml',   #meals
    'fn',   #friends
    'st',   #spiritual
    'hm',   #home
    'ms',   #music
    'fi',   #finance
    'wk',   #work
    'sl',   #sleep
    'pl',   #planning
    'sp',   #sports
    'fm',   #family
    'fx',   #flex
    'tv',   #travel
    'test' #test
]

modes = {}

for i, name in enumerate(mode_ids):
    modes[name] = Time_mode()

mode_names = [
    'read',
    'videogames & shows',
    'hobbies',
    'mental health',
    'physical health',
    'hygiene',
    'meals',
    'friends',
    'spiritual',
    'home',
    'music',
    'finance',
    'work',
    'sleep',
    'planning',
    'sports',
    'family',
    'flex',
    'travel',
    'test'
]

for i, mode in enumerate(modes):
    modes[mode].assign_name(mode_names[i])

In [5]:
#Define the Event class and create the events
for mode in modes:
    modes[mode].clear_events()

class Event:

    def __init__(self, name: str, parent_mode, time_start: datetime, time_end: datetime, location_set:list=None, vague:bool=False):
        
        self.name = name
        if vague:
            self.name = name + " (Vague)"
        
        if time_start > time_end:
            print(f"Error, please fix start and end times")
            raise TypeError("BaseException")
        
        self.time_start=time_start
        self.time_end=time_end

        #parent_mode.print_events()

        if any(x.name == self.name for x in parent_mode.events):
            for i, x in enumerate(parent_mode.events):
                if x.name == self.name:
                    print("Found Duplicate Event:")
                    x.print_event_info()
            return
        
        self.parent_mode = parent_mode
        parent_mode.events.append(self)

        self.location_set = [locations[x] for x in location_set] if location_set else [locations[x] for x in locations.keys()]
        

    def print_event_info(self):
        print(f"Name of Event: {self.name} which occurs from {self.time_start} to {self.time_end}")

def print_all_events():
    for mode in modes:
        print(f"Events for {modes[mode].name}:")
        modes[mode].print_events()
        print("\n")

    
#Create the Events of the current moment
Event("Denver Wellness Appointment", modes["mh"], datetime(2025, 6, 11, 14, 30), datetime(2025, 6, 11, 15, 15), location_set=["Interlocken"])


In [6]:
#Define Daily/Weekly Tasks Classes and initializes the element

for mode in modes:
    modes[mode].clear_daily_tasks()
    modes[mode].clear_weekly_tasks()

class Daily_task:

    def __init__(self, name: str, parent_mode, time_est: timedelta = None, ideal_time: time = None, possible_locations: list = None, need_before: str = None, need_after: str = None, ideal_prd: str = None, ideal_suc: str = None):

        #Sets the name, a str object
        # if not isinstance(name, str):
        #     raise TypeError("AttributeError")
        self.name = name
        
        #Sets the estimated time to completion, a timedelta object
        # if time_est != None and (not isinstance(time_est, (timedelta))):
        #     print(f"type: {type(time_est)}")
        #     raise TypeError("AttributeError")
        self.time_est = time_est

        #Sets the ideat time to complete, a time object
        # if ideal_time != None and not isinstance(ideal_time, time):
        #     raise TypeError("AttributeError")
        self.ideal_time = ideal_time
        
        #Sets the parent mode, a time mode object
        # if not isinstance(parent_mode, Time_mode):
        #     raise TypeError("AttributeError")
        #Ensures daily goal is not a duplicate
        if any(x.name == name for x in parent_mode.daily_tasks):
            for i, x in enumerate(parent_mode.daily_tasks):
                if x.name == name:
                    print("Found Duplicate Goal:")
                    x.print_goal_info()
            return
        self.parent_mode = parent_mode
        parent_mode.daily_tasks.append(self)

        self.location_set = [locations[x] for x in possible_locations] if possible_locations is not None else [locations[x] for x in locations.keys()]

        self.need_before = need_before
        self.need_after = need_after

    def print_goal_info(self):
        print(f"goal name: {self.name} \nparent: {self.parent_mode.name}, which should take {str(self.time_est)}")
        if self.ideal_time != None:
            print(f"Ideally at {str(self.ideal_time)}")
        elif self.ideal_prd != None:
            print(f"Idally just after {self.ideal_prd.name}")
        elif self.ideal_suc != None:
            print(f"Ideally just before {self.ideal_suc.name}")
        else:
            print("At any time")

    def rename(self, name:str):
        self.name=name

class Weekly_task:

    def __init__(self, name: str, parent_mode, time_est: timedelta, ideal_time: time = None, ideal_day:str = None, possible_locations: list = None):
        self.name = name
        self.time_est = time_est
        self.ideal_time = ideal_time if ideal_time else None
        self.ideal_day = ideal_day if ideal_day else None
        self.location_set = [locations[x] for x in possible_locations] if possible_locations else [locations[x] for x in locations.keys()]

        if any(x.name == name for x in parent_mode.weekly_tasks):
            for i, x in enumerate(parent_mode.weekly_tasks):
                if x.name == name:
                    print("Found Duplicate Weekly Task:")
                    x.print_goal_info()
            return
        
        self.parent_mode = parent_mode
        parent_mode.weekly_tasks.append(self)

    def print_goal_info(self):
        print(f"Weekly Task name: {self.name} \nparent: {self.parent_mode.name}, which should take {str(self.time_est)}")
        if self.ideal_time != None:
            print(f"Ideally at {str(self.ideal_time)}")
        elif self.ideal_day != None:
            print(f"Ideally on {self.ideal_day}")
        else:
            print("At any time")



Daily_task("Healthy Sleep", modes["sl"], time_est=timedelta(hours=8), ideal_time=time(hour=4), possible_locations=["Springdale","VG","Estes"])
Daily_task("Reading Quota", modes["rd"], time_est=timedelta(hours=.5), ideal_time=time(hour=17)) 
Daily_task("Fulfilling Hobbies", modes["hb"], time_est=timedelta(hours=.5), ideal_time=time(hour=17))
Daily_task("Wake Up", modes["fx"], time_est=timedelta(hours=.25), ideal_time=time(hour=6), need_before="Healthy Sleep")
Daily_task("Breakfast", modes["ml"], time_est=timedelta(minutes=15), ideal_time=time(hour=7))
Daily_task("Workout", modes["ph"], time_est=timedelta(minutes=30), ideal_time=time(hour=7), possible_locations=["Springdale"], need_after="Morning Hygiene")
Daily_task("Morning Hygiene", modes["hg"], time_est=timedelta(minutes=15), ideal_time=time(hour=7, minute=30), possible_locations=["Springdale", "VG"])
Daily_task("Meditate", modes["mh"], time_est=timedelta(minutes=15), ideal_time=time(hour=6, minute=30))
Daily_task("Journal", modes["mh"], time_est=timedelta(minutes=15), ideal_time=time(hour=7, minute=30), possible_locations=["Springdale"])
Daily_task("Rosary", modes["st"], time_est=timedelta(minutes=15), ideal_time=time(hour=7, minute=45))
Daily_task("Wind Down", modes["fx"], time_est=timedelta(minutes=15), ideal_time=time(hour=23, minute=55), possible_locations=["Springdale", "VG", "Estes"], need_after="Healthy Sleep")
Daily_task("Nightly Hygiene", modes["hg"], time_est=timedelta(minutes=15), ideal_time=time(hour=23, minute=45), possible_locations=["Springdale", "VG", "Estes"])
Daily_task("Plan Next Day", modes["pl"], time_est=timedelta(minutes=15), ideal_time=time(hour=22))
Daily_task("Answer Emails", modes["pl"], time_est=timedelta(minutes=15), ideal_time=time(hour=12))
Daily_task("Lunch", modes["ml"], time_est=timedelta(minutes=15), ideal_time=time(hour=12))
Daily_task("Dinner", modes["ml"], time_est=timedelta(minutes=15), ideal_time=time(hour=18))
Daily_task("Enjoy 15 mins of music", modes["ms"], time_est=timedelta(minutes=15), ideal_time=time(hour=17))
Daily_task("Generally Clean for 15 mins", modes["hm"], time_est=timedelta(minutes=15), possible_locations=["Springdale", "VG", "Estes"], ideal_time=time(hour=17))

Weekly_task("Grocery Shopping", modes["ml"], time_est=timedelta(hours=1), ideal_time=time(hour=8), ideal_day="Sun", possible_locations=["North 28th"])
Weekly_task("Meal Prep", modes["ml"], time_est=timedelta(hours=2), ideal_time=time(hour=10), ideal_day="Sun", possible_locations=["Springdale"])



In [7]:
#Dfine Task class and initializes the tasks

for mode in modes:
    modes[mode].clear_tasks()

class Task:

    def __init__(self, name:str, parent_mode:Time_mode, time_est:timedelta, parent_task:list = None, deadline:datetime = None, possible_locations:list = None):

        #Sets the name, a str object
        self.name = name
        
        #Sets the estimated time to completion, a timedelta object
        self.time_est = time_est
        
        if not isinstance(parent_mode, Time_mode):
            print(f"Error: {parent_mode} is not a Time_mode; it is {type(parent_mode)}")
            raise TypeError("AttributeError")

        #Sets the parent mode, a time mode object

        if deadline != None and not isinstance(deadline, datetime):
            raise TypeError("AttributeError")
        if deadline != None and deadline > datetime.now() + timedelta(days=1):
            self.deadline = deadline
        elif deadline != None:
            print("Error: Deadline must be at least one day in the future")
            raise TypeError("AttributeError")
        
        self.deadline = deadline

        #Ensures long term goal is not a duplicate
        if any(x.name == name for x in parent_mode.tasks):
            for i, x in enumerate(parent_mode.tasks):
                if x.name == name:
                    print("Found Duplicate Long Term Goal:")
                    x.print_goal_info()
            return
        
        self.parent_mode = parent_mode
        
        parent_mode.tasks[self] = self.time_est

        if parent_task != None and not isinstance(parent_task, list):
            print(f"Error: {parent_task} is not a List; it is {type(parent_task)}")
            raise TypeError("AttributeError")
        
        if parent_task is not None:
            if not isinstance(parent_task, list):
                raise TypeError("AttributeError: parent_task must be a list of str which direct to a Task object")
            
            if parent_task[0] not in modes:
                raise ValueError("AttributeError: parent_task[0] not in modes")
            
            for task in modes[parent_task[0]].tasks:
                if task.name == parent_task[1]:
                    self.parent_task = task
                    break
            if not hasattr(self, 'parent_task'):
                raise ValueError("AttributeError: parent_task not in modes.tasks")
        else:
            self.parent_task = None

        self.parent_task.kid_task.append(self) if parent_task != None else None
        self.kid_task = []

        if possible_locations != None and not isinstance(possible_locations, list):
            print(f"Error: {possible_locations} is not a list; it is {type(possible_locations)}")
            raise TypeError("AttributeError")
        
        self.location_set = [locations[x] for x in possible_locations] if possible_locations is not None else [locations[x] for x in locations.keys()]   

    def print_goal_info(self):
        print(f"Long Term Goal: {self.name} \nParent Mode: {self.parent_mode.name}, which should take {str(self.time_est)}")
        if self.deadline != None:
            print(f"Deadline: {self.deadline}")
        else:
            print("No deadline set")

Task("Transfer Tasks to Python", modes["pl"], time_est=timedelta(hours=1), deadline=None)
Task("Understand ROM Flash", modes["wk"], time_est=timedelta(hours=6), deadline=None)
Task("Secure Teaching Interview", modes["wk"], time_est=timedelta(hours=6), deadline=datetime(2025, 7, 17, 12, 0))

Task("Interview at East", modes["wk"], time_est=timedelta(hours=1), possible_locations=["East Colfax"])
Task("Pick Up Luggage Rack", modes["hb"], time_est=timedelta(hours=.5), possible_locations=["Euro Moto Electric"])
Task("Secure ROM Flashing Rig To Phone", modes["wk"], time_est=timedelta(hours=3), parent_task=["wk", "Understand ROM Flash"])

Task("Test-Catchall for Reading", modes["rd"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Videogames", modes["vs"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Hobbies", modes["hb"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Mental Health", modes["mh"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Physical Health", modes["ph"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Hygiene", modes["hg"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Meals", modes["ml"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Friends", modes["fn"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Spiritual", modes["st"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Home", modes["hm"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Music", modes["ms"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Finance", modes["fi"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Work", modes["wk"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Sleep", modes["sl"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Planning", modes["pl"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Sports", modes["sp"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Family", modes["fm"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Flex", modes["fx"], time_est=timedelta(hours=1000))
Task("Test-Catchall for Travel", modes["tv"], time_est=timedelta(hours=1000))


#Task("Break 1", modes["wk"], time_est=timedelta(minutes=15), deadline=datetime(2025, 1, 1))
#Task("Break 2", modes["wk"], time_est=timedelta(minutes=15), parent_task=["wk", "Pick Up Luggage Rack"], deadline=datetime(2025, 6, 15, 12, 0))


In [8]:
def find_differences(activities, insert_index):
    center = insert_index + 1
    rolled = np.roll(activities[:,1], -center)  # activities[:,1] becomes [3, 4, 5, 1, 2]
    trimmed = rolled[:-1]
    sum_left = np.cumsum(trimmed[::-1])[::-1]
    sum_right = np.cumsum(trimmed)
    minimums = np.min([sum_left, sum_right], axis=0)
    outrolled = np.concatenate(([0], minimums, [0]))
    outunrolled = np.roll(outrolled, center)  # outrolled becomes [1, 2, 3, 4, 5]
    outunrolled = np.delete(outunrolled, np.argmax(outunrolled))  # Remove the maximum value

    if outunrolled[0] <= outunrolled[-1]:
        outunrolled = np.concatenate((outunrolled, [outunrolled[0]]))
    else:
        outunrolled = np.concatenate(([outunrolled[-1]], outunrolled))

    return outunrolled


def find_where_none(activities):
    result = np.full(activities.shape[0]+1, False, dtype=object)  # Create a boolean array of the same length as activities with an extra element
    
    if activities[0, 0] is None:
        result[0] = activities[0, 1]

    for i in range(activities.shape[0])[1:]:
        if activities[i-1, 0] is None:
            result[i] = activities[i-1, 1]

        elif activities[i, 0] is None:
            result[i] = activities[i, 1]

    if activities[-1, 0] is None:
        result[-1] = activities[-1, 1]

    return result

def find_where_task(activities, task):
    result = np.full(activities.shape[0]+1, False, dtype=object)  # Create a boolean array of the same length as activities with an extra element

    if activities[0, 0] is task:
        result[0] = activities[0, 1]

    for i in range(activities.shape[0])[1:]:
        if activities[i-1, 0] is task:
            result[i] = activities[i-1, 1]

        elif activities[i, 0] is task:
            result[i] = activities[i, 1]

    if activities[-1, 0] is task:
        result[-1] = activities[-1, 1]

    return result

def get_parent_task(activities, tasks, task):
    if task.need_before is not None:
        parent_task = [t for t in tasks if t.name == task.need_before][0]

    elif task.need_after is not None:
        parent_task = [t for t in tasks if t.name == task.need_after][0]

    if parent_task is None:
        raise ValueError(f"Parent task for {task.name} not found in tasks.")
    return parent_task

def get_parent_indeces(activities, parent_task):
    """Find the indices of the parent task in the activities array."""
    return [i for i, x in enumerate(activities[:, 0]) if x is parent_task]

def time_between_indexes(activities, start_index, end_index):
    if start_index < end_index:    
        return np.sum(activities[start_index+1:end_index, 1])
    
    elif start_index > end_index:
        return np.sum(activities[start_index+1:, 1]) + np.sum(activities[:end_index, 1])
    
    else: return 0

def parent_indeces_compare(activities, parent_task):
    """Compare the indices of the parent task in the activities array."""
    indeces = get_parent_indeces(activities, parent_task)
    return np.vstack((indeces, np.roll(indeces, -1)))

def parent_indeces_differences(activities, parent_task):
    """Calculate the time differences between the indices of the parent task in the activities array."""
    parent_indices = parent_indeces_compare(activities, parent_task)
    return [time_between_indexes(activities, parent_indices[0][i], parent_indices[1][i]) for i in range(len(parent_indices[1]))]


def get_anchor_index(activities, parent_task, task, is_self=False, last_direction=None):

    if last_direction is not None and last_direction not in ['left', 'right']:
        raise ValueError("last_direction must be either 'left' or 'right'")

    """Determine the anchor index based on the task's need_before or need_after attributes."""

    if is_self and last_direction is None:
        differences_eligible, direction = get_differences_eligible(activities, task, is_self=True)
        if direction == "left":
            return np.argmin(differences_eligible) - 1
        else:
            return np.argmin(differences_eligible)

    parent_indices = parent_indeces_compare(activities, parent_task)
    differences = parent_indeces_differences(activities, parent_task)
    
    if last_direction == "left":
        return parent_indices[1][np.argmin(differences)]    #Runs in the particular case for the peel start from orphan
    elif last_direction == "right":
        return parent_indices[0][np.argmin(differences)]


    if task.need_before is not None:
        return parent_indices[0][np.argmax(differences)]
    else:
        return parent_indices[1][np.argmax(differences)]
        
    
def get_insert_index(activities, parent_task, task):
    print("Get insert index runs")
    anchor_index = get_anchor_index(activities, parent_task, task)
    if task.need_before is not None:
        return anchor_index + 1
    else:
        return anchor_index

def gen_check_indeces(activities, parent_task, task, is_self=False):
    anchor_index = get_anchor_index(activities, parent_task, task, is_self=is_self)

    check_index_left = anchor_index
    check_index_right = anchor_index

    while activities[check_index_left, 0] in [parent_task, task]:
        check_index_left = (check_index_left - 1) % len(activities[:, 0])

    while activities[check_index_right, 0] in [parent_task, task]:
        check_index_right = (check_index_right + 1) % len(activities[:, 0])
    
    return check_index_left, check_index_right


def search_for_movable(activities, tasks, task, is_self=False):
    """Search for the first movable activity to the left or right of the anchor index."""
    
    
    if not is_self:
        anchor_index = get_anchor_index(activities, get_parent_task(activities, tasks, task), task, is_self=is_self)
        left_searching, right_searching = gen_check_indeces(activities, get_parent_task(activities, tasks, task), task)
    else:
        anchor_index = get_anchor_index(activities, task, task, is_self=True)
        left_searching, right_searching = gen_check_indeces(activities, task, task, is_self=True)   

    while left_searching != right_searching:
        if activities[left_searching, 0] is None or (not hasattr(activities[left_searching, 0], 'ideal_time') and not isinstance(activities[left_searching, 0], Event)):
            return left_searching, "left", activities[left_searching, 0]
        elif activities[right_searching, 0] is None or (not hasattr(activities[right_searching, 0], 'ideal_time') and not isinstance(activities[right_searching, 0], Event)):
            return right_searching, "right", activities[right_searching, 0]
        
        else:
            if time_between_indexes(activities, left_searching, anchor_index) <= time_between_indexes(activities, anchor_index, right_searching):
                left_searching = (left_searching - 1) % len(activities[:, 0])
            else:
                right_searching = (right_searching + 1) % len(activities[:, 0])
    
    raise ValueError("AttributeError: No movable activity found to the left or right of the anchor index.")

def search_for_none(activities, start_index, direction="left"):
    if direction == "left":
        index = start_index
        while activities[index, 0] is not None:
            index = (index - 1) % len(activities[:, 0])
            if index == start_index:
                return None
        return index
    
    elif direction == "right":
        index = start_index
        while activities[index, 0] is not None:
            index = (index + 1) % len(activities[:, 0])
            if index == start_index:
                return None
        return index
    
    raise ValueError("AttributeError: Invalid direction specified. Use 'left' or 'right'.")

def insert_task(activities, tasks, task, is_self=False):

    if is_self:
        insert_index = get_anchor_index(activities, task, task, is_self=True)
        movable_index, direction, movable_task = search_for_movable(activities, tasks, task, is_self=True)

    else:
        insert_index = get_insert_index(activities, get_parent_task(activities, tasks, task), task)
        movable_index, direction, movable_task = search_for_movable(activities, tasks, task)

    if tasks[task].total_seconds() <= 0:
        return activities, 0, None

    task_need = tasks[task].total_seconds() / 60
    movable_accomodation = activities[movable_index, 1]
    none_index = search_for_none(activities, movable_index, direction)
    none_space = activities[none_index, 1]

    print(min(task_need, movable_accomodation, none_space))

    if task_need == min(task_need, movable_accomodation, none_space):
        result = np.insert(activities, insert_index, [task, task_need, None], axis=0)
        tasks[task] -= timedelta(minutes=task_need)
        time = task_need

    elif movable_accomodation == min(task_need, movable_accomodation, none_space):
        result = np.insert(activities, insert_index, [task, movable_accomodation, None], axis=0)
        tasks[task] -= timedelta(minutes=movable_accomodation)
        time = movable_accomodation

    else:
        print(none_space)
        result = np.insert(activities, insert_index, [task, none_space, None], axis=0)
        tasks[task] -= timedelta(minutes=int(none_space))
        time = none_space

    return result, time, movable_task


def search_for_nearest_task_like(activities, start_index, task_like=None, direction="both"):
    if direction == "both":
        left_index = start_index
        right_index = start_index
        
        while True:
            if activities[left_index, 0] == task_like or (task_like is None and activities[left_index, 0] is None):
                return left_index, "left"
            
            if activities[right_index, 0] == task_like or (task_like is None and activities[right_index, 0] is None):
                return right_index, "right"
            
            if time_between_indexes(activities, left_index, start_index) <= time_between_indexes(activities, start_index, right_index):
                left_index = (left_index - 1) % len(activities[:, 0])
            else:
                right_index = (right_index + 1) % len(activities[:, 0])
            
            if left_index == right_index:
                raise ValueError("AttributeError: No task-like activity found in either direction.")
        
    elif direction == "left":
        index = start_index
        while True:
            if activities[index, 0] == task_like or (task_like is None and activities[index, 0] is None):
                return index, "left"
            index = (index - 1) % len(activities[:, 0])
            if index == start_index:
                raise ValueError("AttributeError: No task-like activity found in the left direction.")
            
    elif direction == "right":
        index = start_index
        while True:
            if activities[index, 0] == task_like or (task_like is None and activities[index, 0] is None):
                return index, "right"
            index = (index + 1) % len(activities[:, 0])
            if index == start_index:
                raise ValueError("AttributeError: No task-like activity found in the right direction.")
        
def shift_movable(activities, movable_index, destination_index, move_time, direction="left"):
    if direction not in ["left", "right"]:
        raise ValueError("AttributeError: Direction must be 'left' or 'right'.")
    
    if movable_index == destination_index:
        activities[movable_index, 1] -= move_time
        return activities

    if move_time > activities[movable_index, 1]:
        raise ValueError("AttributeError: Move time exceeds the duration of the movable activity.")
    if move_time < 0:
        raise ValueError("AttributeError: Move time cannot be negative.")
    if move_time > activities[destination_index, 1]:
        raise ValueError("AttributeError: Move time exceeds the duration of the destination activity.")
    
    moving_task = activities[movable_index, 0]

    before = activities[:movable_index, :].copy()
    after = activities[movable_index + 1:, :].copy()
    
    if direction == "left":
        activities = np.vstack((before, [moving_task, activities[movable_index, 1] - move_time, None], after))
        
    else:
        activities = np.vstack((before, [moving_task, activities[movable_index, 1] - move_time, None], after))

    #Insert the popped task into the destination index

    destination_index = search_for_nearest_task_like(activities, destination_index, task_like=None, direction="both")[0]

    if activities[destination_index, 0] is not None:
        raise AttributeError("Cannot shift movable activity to a destination that is already occupied by another task.")
        
    before = activities[:destination_index, :].copy()
    after = activities[destination_index + 1:, :].copy()

    if direction == "left":
        activities = np.vstack((before, [None, activities[destination_index, 1] - move_time, None], [moving_task, move_time, None], after))

    else:
        activities = np.vstack((before, [moving_task, move_time, None], [None, activities[destination_index, 1] - move_time, None], after))

    return activities


def peel_activities(activities, start_index, end_index, direction="left", time_inserted=0):
    if direction not in ["left", "right"]:
        raise ValueError("Direction must be either 'left' or 'right'.")

    i_index = start_index

    peeled_queue = []
    event_queue = []

    while i_index != end_index:

        if activities[i_index, 0] is None:
            print(i_index, end_index, start_index, direction)
            raise ValueError("There should not have been a None value in this sequence")
        
        if not isinstance(activities[i_index, 0], Event):
            peeled_queue.append([activities[i_index, 0], activities[i_index, 1]])
            activities[i_index, 0] = None

        else:
            if direction == "left":
                event_time = np.sum(activities[:i_index, 1])
                event_queue.append([activities[i_index, 0], event_time, activities[i_index, 1]])
                activities[i_index, 0] = None
            else:
                event_time = np.sum(activities[:i_index, 1]) - time_inserted
                event_queue.append([activities[i_index, 0], event_time, activities[i_index, 1]])
                activities[i_index, 0] = None

        i_index = (i_index - 1) % len(activities[:, 0]) if direction == "left" else (i_index + 1) % len(activities[:, 0])

    i_index = 0

    return activities, peeled_queue, event_queue

def insert_events(activities, event_queue):

    for event, event_time, duration in event_queue:

        if event_time < 0:
            raise ValueError("Event time cannot be negative.")
        
        i = 0
        while i < len(activities[:,0])-1 and np.sum(activities[:i+1, 1]) < event_time:
            i += 1

        split_time = event_time - np.sum(activities[:i, 1])

        before = activities[:i, :].copy()
        after = activities[i+1:, :].copy()

        activities = np.vstack((before, [None, split_time, None], [None, activities[i, 1] - split_time, None], after))

        activities = np.insert(activities, i+1, [event, duration, None], axis=0)

        delete_time = duration
        i = (i + 2) % len(activities[:, 0])  # Move to the next index after inserting the event
        while delete_time > 0:

            if activities[i, 0] is not None:
                raise AttributeError("Cannot delete a non-None activity while inserting an event.")

            if activities[i, 1] <= delete_time:
                delete_time -= activities[i, 1]
                activities[i, 1] = 0
            else:
                activities[i, 1] -= delete_time
                delete_time = 0
            
            i = (i + 1) % len(activities[:, 0])

    return activities



def unpeel_activities(activities, peeled_queue, start_index, direction="left"):
    if direction not in ["left", "right"]:
        raise ValueError("Direction must be either 'left' or 'right'.")
    
    i_index = start_index

    for task, duration in peeled_queue:

        while duration > 0:
            if activities[i_index, 0] is not None:
                if direction == "left":
                   i_index = (i_index - 1) % len(activities[:, 0])
                else:
                    i_index = (i_index + 1) % len(activities[:, 0])
                continue
            
            if activities[i_index, 1] <= duration:
                duration -= activities[i_index, 1]
                activities[i_index, 0] = task
            
            else:
                before = activities[:i_index, :].copy()
                after = activities[i_index + 1:, :].copy()
                if direction == "left":
                    activities = np.vstack((before, [None, activities[i_index, 1] - duration, None], [task, duration, None], after))
                else:
                    activities = np.vstack((before, [task, duration, None], [None, activities[i_index, 1] - duration, None], after))
        
                duration = 0

            if direction == "left":
                i_index = (i_index - 1) % len(activities[:, 0])
            else:
                i_index = (i_index + 1) % len(activities[:, 0])


    return activities

def insert_with_parent(activities, tasks, task):
    if np.sum([x[1] for x in activities if x[0] is None]) < tasks[task].total_seconds() / 60:
        raise ValueError("AttributeError: Not enough time in activities to distribute task")
    
    if tasks[task].total_seconds() / 60 <= 0:
        return activities
    
    activities, time_inserted, movable_task = insert_task(activities, tasks, task) #Insert the task where it should go
    start_index = get_insert_index(activities, get_parent_task(activities, tasks, task), task)  #Re-gather the index of the inserted task
    found_index, found_direction = search_for_nearest_task_like(activities, start_index, movable_task, direction="both")    #Re-gather the movable task's index and direction
    found_index_none, found_direction_none = search_for_nearest_task_like(activities, found_index, task_like=None, direction=found_direction)   #Find the nearest None task to the movable task
    activities = shift_movable(activities, found_index, found_index_none, time_inserted, direction=found_direction_none)    #Move the activity that's allowed to move to the available none space
    start_index = get_insert_index(activities, get_parent_task(activities, tasks, task), task)  #Re-gather the start index of the task that's been inserted
    if found_direction == "left":
        start_index -= 1 #This ensures that the first activity peeled is the same one we insert
    end_index = search_for_nearest_task_like(activities, start_index, task_like=movable_task, direction=found_direction)[0] #Re-gather the end of the window for activities to be shifted
    activities, peeled_queue , event_queue = peel_activities(activities, start_index, end_index, direction=found_direction, time_inserted=time_inserted)    #We peel the events into tasks and events
    activities = insert_events(activities, event_queue) #We re-insert the events into the activities
    start_index = get_insert_index(activities, get_parent_task(activities, tasks, task), task)  #Re-gather the start index of the task that's been inserted
    if found_direction == "left":
        start_index -= 1 #This ensures that the first activity peeled is the same one we insert
    activities = unpeel_activities(activities, peeled_queue, start_index, direction=found_direction)   #we unpeel the tasks back into the activities
    activities = clear_zeros(activities)

    return activities

def insert_orphan(activities, tasks, task):
    if np.sum([x[1] for x in activities if x[0] is None]) < tasks[task].total_seconds() / 60:
        raise ValueError("AttributeError: Not enough time in activities to distribute task")
    
    if tasks[task].total_seconds() / 60 <= 0:
        return activities

    activities, time_inserted, movable_task  = insert_task(activities, tasks, task, is_self=True) #Insert the task where it should go
    start_index = get_anchor_index(activities, task, task, is_self=True)  #Re-gather the index of the inserted task
    found_index, found_direction = search_for_nearest_task_like(activities, start_index, movable_task, direction="both")    #Re-gather the movable task's index and direction
    found_index_none, found_direction_none = search_for_nearest_task_like(activities, found_index, task_like=None, direction=found_direction)   #Find the nearest None task to the movable task
    activities = shift_movable(activities, found_index, found_index_none, time_inserted, direction=found_direction_none)    #Move the activity that's allowed to move to the available none space
    start_index = get_anchor_index(activities, task, task, is_self=True, last_direction=found_direction)  #Re-gather the start index of the task that's been inserted
    end_index = search_for_nearest_task_like(activities, start_index, task_like=movable_task, direction=found_direction)[0] #Re-gather the end of the window for activities to be shifted
    activities, peeled_queue , event_queue = peel_activities(activities, start_index, end_index, direction=found_direction, time_inserted=time_inserted)    #We peel the events into tasks and events
    activities = insert_events(activities, event_queue) #We re-insert the events into the activities
    start_index = get_anchor_index(activities, task, task, is_self=True)  #Re-gather the start index of the task that's been inserted
    if direction == "left":
        start_index = (start_index - 1) % len(activities[:, 0]) #This ensures that the first activity peeled is the same one we insert
    activities = unpeel_activities(activities, peeled_queue, start_index, direction=found_direction)   #we unpeel the tasks back into the activities
    activities = clear_zeros(activities)

    return activities

def get_differences_eligible(activities, task, is_self=False):
    if not hasattr(task, 'ideal_time'):
        raise AttributeError("AttributeError: Task must have an ideal_time attribute.")

    ideal_time = task.ideal_time.hour * 60 + task.ideal_time.minute  # Convert ideal_time to minutes

    differences = np.concatenate(([0], np.cumsum(activities[:, 1])))
    differences_left = (differences - ideal_time) % 1440
    differences_right = (ideal_time - differences) % 1440
    differences_stack = np.vstack((differences_left, differences_right))
    
    differences = np.min(differences_stack, axis=0)


    if not is_self:
        none_present = find_where_none(activities) != False
        if not any(none_present):
            raise AttributeError("AttributeError: No None values found in activities.")

        differences_eligible = [differences[i] if none_present[i] else 1440 for i in range(len(differences))]

    else:
        task_present = find_where_task(activities, task) != False
        if not any(task_present):
            raise AttributeError("AttributeError: No task values found in activities.")
        
        differences_eligible = [differences[i] if task_present[i] else 1440 for i in range(len(differences))]

    if np.argmin(np.min(differences_stack, axis=1)) == 0:
        direction = "left"
    else:
        direction = "right"

    return differences_eligible, direction  #I'm pretty sure that in this context, direction means the side of the slot you want to insert.

def insert_at_none(activities, tasks, task, insert_index, direction = "left"):
    if direction not in ["left", "right"]:
        raise AttributeError("AttributeError: Direction must be either 'left' or 'right'.")
    
    if not hasattr(task, 'ideal_time'):
        raise AttributeError("AttributeError: Task must have an ideal_time attribute.")

    if direction == 'left':
        if insert_index != 0 and activities[insert_index - 1, 0] is None:
            direction = 'left'
            insert_index -= 1
        else:
            direction = 'right'
    
    elif direction == 'right':
        if insert_index != len(activities) and activities[insert_index, 0] is None:
            direction = 'right'
        else:  
            direction = 'left'

    if activities[insert_index, 0] is not None:
        raise AttributeError("AttributeError: Cannot insert at an index that is already occupied by a task.")
    
    before = activities[:insert_index, :].copy()
    after = activities[insert_index + 1:, :].copy()

    insert_time = min(tasks[task].total_seconds() / 60, activities[insert_index, 1])

    if insert_time == activities[insert_index, 1]:
        activities[insert_index, 0] = task
        tasks[task] -= timedelta(minutes=insert_time)
        return activities

    else:
        if direction == 'left':
            activities = np.vstack((before, [None, activities[insert_index, 1] - insert_time, None], [task, insert_time, None], after))
        else:
            activities = np.vstack((before, [task, insert_time, None], [None, activities[insert_index, 1] - insert_time, None], after))
        tasks[task] -= timedelta(minutes=insert_time)

    return activities
    
def insert_anywhere(activities, tasks, task, direction='left'):
    if direction not in ["left", "right"]:
        raise AttributeError("AttributeError: Direction must be either 'left' or 'right'.")

    insert_time = tasks[task].total_seconds() / 60
    
    if insert_time > np.sum([x[1] for x in activities if x[0] is None and x[1] > 0]):
        raise ValueError("AttributeError: Not enough time in activities to distribute task")
    
    if direction == 'left':
        insert_index = np.min([i for i, x in enumerate(activities[:, 0]) if x is None and activities[i, 1] > 0])
    else:
        insert_index = np.max([i for i, x in enumerate(activities[:, 0]) if x is None and activities[i, 1] > 0])

    if activities[insert_index, 1] < insert_time:
        activities[insert_index, 0] = task
        tasks[task] -= timedelta(minutes=activities[insert_index, 1])
    else:
        before = activities[:insert_index, :].copy()
        after = activities[insert_index + 1:, :].copy()
        if direction == 'left':
            activities = np.vstack((before, [task, insert_time, None], [None, activities[insert_index, 1] - insert_time, None], after))
        else:
            activities = np.vstack((before, [None, activities[insert_index, 1] - insert_time, None], [task, insert_time, None], after))
        tasks[task] = timedelta(minutes=0)

    return activities

def clear_zeros(activities):
    """Remove rows with zero duration from the activities array."""
    eliminate = []
    for i in range(activities.shape[0]):
        if activities [i, 1] <= 0 and type(activities[i, 0]) != Event:
            eliminate.append(i)
    return np.delete(activities, eliminate, axis=0)

    



    

In [9]:
#Define Day Class

class Day:

    def __init__(self, day_date:date, flex_time:timedelta = timedelta(hours=1.5), travel_time:timedelta = timedelta(hours=1), suggested_activities:int=50):
        print("Creating new day object")
        
        if not isinstance(day_date, date):
            raise TypeError("AttributeError: date must be a date object")
        
        self.date = day_date

        self.daily_ideals = {}
        self.daily_ideals_set = False

        self.events = []
        self.events_set = False

        self.tasks = {}
        self.tasks_set = False

        for mode in modes:
            for goal in modes[mode].daily_tasks:
                self.tasks[goal] = goal.time_est
        self.tasks_set = True

        self.flex_time = flex_time
        self.travel_time = travel_time

        self.locked_in_dist = np.zeros(len(modes), dtype=int)
        self.locked_in_dist_set = False

        self.locked_in_ok = False

        self.proposed_dist = np.zeros(len(modes), dtype=int)
        self.proposed_dist_set = False

        self.activities_list = []
        self.activities_list_set = False

        self.additional_tasks_set = False

    def clear_event(self, name:str=None, v:bool=False):
        if name is None:
            self.events = []
        else:
            print(f"Trying to find {name} in {self.date} events")
            for i in range(len(self.events)):
                if self.events[i].name == name:
                    print(f"Found {name} in {self.date} events")
                    del self.events[i]
                    return
            else:
                if v: print(f"Could not find {name} in {self.date} events")


    def set_ideals(self, ideals:dict):
        if not isinstance(ideals, dict):
            raise TypeError("AttributeError: ideals must be a dict")
        
        self.daily_ideals = ideals
        self.daily_ideals_set = True

    def set_locked_in_dist(self):
        if not self.daily_ideals_set:
            raise ValueError("AttributeError: daily_ideals must be set before setting locked_in_dist")
        
        if any([not self.daily_ideals_set, not self.events_set, not self.tasks_set]):
            raise ValueError("AttributeError: daily_ideals, events, and tasks must be set before setting locked_in_dist")
        
        dist_temp = {modes[mode]: int(0) for mode in modes}

        #Update for daily locked in values, beginning with tasks (will include ideals at the daily level, we decided)

        for task in self.tasks:
            if task.ideal_time is not None:
                if task.parent_mode not in dist_temp:
                    raise ValueError(f"AttributeError: {task.parent_mode} not in dest_temp")
                
                dist_temp[task.parent_mode] += int(task.time_est.total_seconds() / 60)

        #Update for events
        for event in self.events:
            if event.time_start is not None and event.time_end is not None:
                if event.parent_mode not in dist_temp:
                    raise ValueError(f"AttributeError: {event.parent_mode} not in dest_temp")
                
                dist_temp[event.parent_mode] += int((event.time_end - event.time_start).total_seconds() / 60)

        self.locked_in_dist = np.array([dist_temp[modes[mode]] for mode in modes], dtype=int)
        self.locked_in_dist_set = True

        if not self.locked_in_ok:
            total_minutes_locked_in = np.sum(self.locked_in_dist)

            if total_minutes_locked_in > (24 * 60):
                print(f"Warning: Total minutes locked in ({total_minutes_locked_in}) exceeds 24 hours. Please check your daily ideals, events, and tasks.")
            else:
                self.locked_in_ok = True

    def set_proposed_dist_lean(self, minutes_to_distribute:np.ndarray):
        if not isinstance(minutes_to_distribute, np.ndarray):
            raise TypeError("AttributeError: proposed_dist must be a numpy array")
        
        if self.proposed_dist_set:
            print("Propoed dist already set; skipping.")
            return

        if not self.locked_in_dist_set:
            print("Error: locked_in_dist must be set before setting proposed_dist")
            raise ValueError("AttributeError: locked_in_dist must be set before setting proposed_dist")
        
        if not self.daily_ideals_set:
            print("Error: daily_ideals must be set before setting proposed_dist")
            raise ValueError("AttributeError: daily_ideals must be set before setting proposed_dist")
        
        self.proposed_dist = self.locked_in_dist.copy() #Start from where we know we are at

        daily_ideals_tf = self.daily_ideals.copy()  #This will let us know how many minutes to account for from minutes_to_distribute -- updates with daily flex and travel attributes
        for mode in modes:
            if mode == "fx" and daily_ideals_tf[mode] < self.flex_time:
                daily_ideals_tf[mode] = self.flex_time
                print(f"Setting accurate daily flex time")
            elif mode == "tv" and daily_ideals_tf[mode] < self.travel_time:
                daily_ideals_tf[mode] = self.travel_time
                print(f"Setting accurate daily travel time")

        #Then we add; just the daily ideals

        proposed_dist_before = self.proposed_dist.copy()    #This will let us know how many minutes to account for from minutes_to_distribute
        daily_ideals_np = np.array([int(daily_ideals_tf[mode].total_seconds() / 60) for mode in modes])

        for i, mode in enumerate(modes):
                if not isinstance(self.daily_ideals[mode], timedelta):
                    raise TypeError("AttributeError: daily_ideals must be a dict of int")
                
                time_remaining = 60*24 - np.sum(self.proposed_dist)

                if time_remaining < 0:
                    self.proposed_dist = proposed_dist_before
                    raise ValueError("AttributeError: time_remaining cannot be negative; track down what went badly.")

                to_add = min([time_remaining, max([0, daily_ideals_np[i] - self.locked_in_dist[i]]), minutes_to_distribute[i]])

                # print(f"Adding {to_add} minutes to {mode} from daily ideals at date {self.date}")

                self.proposed_dist[i] += to_add

        minutes_to_distribute -= (self.proposed_dist - proposed_dist_before) #This is the amount of time we have left to distribute

    def set_proposed_dist_greedy(self, minutes_to_distribute:np.ndarray):
        if not isinstance(minutes_to_distribute, np.ndarray):
            raise TypeError("AttributeError: proposed_dist must be a numpy array")
        
        if self.proposed_dist_set:
            print("Propoed dist already set; skipping.")
            return

        if not self.locked_in_dist_set:
            print("Error: locked_in_dist must be set before setting proposed_dist")
            raise ValueError("AttributeError: locked_in_dist must be set before setting proposed_dist")
        
        if not self.daily_ideals_set:
            print("Error: daily_ideals must be set before setting proposed_dist")
            raise ValueError("AttributeError: daily_ideals must be set before setting proposed_dist")
        
        proposed_dist_before = self.proposed_dist.copy()    #This will let us know how many minutes to account for from minutes_to_distribute

        for i, mode in enumerate(modes):
            if not isinstance(self.daily_ideals[mode], timedelta):
                raise TypeError("AttributeError: daily_ideals must be a dict of int")
            
            time_remaining = 60*24 - np.sum(self.proposed_dist)

            if time_remaining < 0:
                self.proposed_dist = proposed_dist_before
                raise ValueError("AttributeError: time_remaining cannot be negative; track down what went badly.")

            to_add = min([time_remaining, minutes_to_distribute[i]])

            self.proposed_dist[i] += to_add

        minutes_to_distribute -= (self.proposed_dist - proposed_dist_before) #This is the amount of time we have left to distribute

        if np.sum(minutes_to_distribute) < 0:
            raise AttributeError("AttributeError: minutes_to_distribute cannot be negative; track down what went badly.")
        
        if np.sum(self.proposed_dist) != (24 * 60):
            self.proposed_dist = proposed_dist_before
            raise ValueError("AttributeError: total minutes proposed is not 24 hours; track down what went badly.")

        self.proposed_dist_set = True

    # def list_all_activities(self, week):
        
    #     if not self.proposed_dist_set:
    #         print("Error: proposed_dist must be set before listing all activities")
    #         raise ValueError("AttributeError: proposed_dist must be set before listing all activities")
        
    #     if self.activities_list_set:
    #         print("Activities list already set; skipping.")
    #         return
        
    #     #Returns a positive int representing the total number of choices we have for things to do.
    #     total_activities = 0
    #     #We begin with a dict of daily tasks
    #     for task in self.tasks:
    #         if not isinstance(task, Daily_task):
    #             raise TypeError("AttributeError: task must be a Daily_task object")
            
    #         if self.tasks[task] > timedelta(minutes=0):
    #             print(f"Adding {task.name} to activities list with time estimate {self.tasks[task]}")
    #             self.activities_list.append(task)

    #     #Then we add the weekly tasks
    #     for task in week.tasks:
    #         if not isinstance(task, Weekly_task) and not isinstance(task, Task):
    #             raise TypeError(f"AttributeError: task {task.name} must be a Weekly_task object, not {type(task)}")
            
    #         if week.tasks[task] > timedelta(minutes=0):
    #             print(f"Adding {task.name} to activities list with time estimate {week.tasks[task]}")
    #             self.activities_list.append(task)

    #     #All the events this day needs to care about are the ones in its own object
    #     for event in self.events:
    #         if not isinstance(event, Event):
    #             raise TypeError("AttributeError: event must be an Event object")

    #         print(f"Adding {event.name} to activities list with time estimate {(event.time_end - event.time_start).total_seconds() / 60}")
    #         self.activities_list.append(event)


    #     #And now we add ALL the other tasks under the sun!
    #     for mode in modes:
    #         for task in modes[mode].tasks:
    #             if not isinstance(task, Task):
    #                 raise TypeError("AttributeError: task must be a Task object")
                
    #             if task not in week.tasks:
    #                 if modes[mode].tasks[task] > timedelta(minutes=0):
    #                     print(f"Adding {task.name} to activities list with time estimate {modes[mode].tasks[task]}")
    #                     self.activities_list.append(task)
    #                     total_activities += 1

    #             else:
    #                 print(f"Skipping {task.name} as it is already in the week's tasks, given its deadline of {task.deadline}.")

    #     self.activities_list_set = True

    #Don't believe this is going to be necessary anymore.
    
    # def generate_events_row(self):
    #     if self.events_row_list_set:
    #         print("Events row list already set; skipping.")
    #         return
        
    #     if not self.activities_list_set:
    #         print("Error: activities_list must be set before generating events row")
    #         raise ValueError("AttributeError: activities_list must be set before generating events row")
        
    #     #This will be a list of lists, where each sublist is a row of the event vector

    #     for i, activity in enumerate(self.activities_list):
    #         if isinstance(activity, Event):
    #             self.events_row_list.append([i, (activity.time_end - activity.time_start).total_seconds() / 60])

    #     self.events_row_list_set = True

    
    #Need to change the print to take events.
    
    def print_activities(self, activities_plan):
        # if not self.activities_list_set:
        #     print("Error: activities_list must be set before printing activities")
        #     raise ValueError("AttributeError: activities_list must be set before printing activities")

        # if not self.events_row_list_set:
        #     print("Error: events_row_list must be set before printing activities")
        #     raise ValueError("AttributeError: events_row_list must be set before printing activities")
        
        time_counter = int(0) #Minutes since midnight

        for i in range(activities_plan.shape[0]):

            print(f"{time(hour=(time_counter%1440) // 60, minute=time_counter%60)}: {activities_plan[i, 0].name if hasattr(activities_plan[i,0], 'name') else "None"} ({activities_plan[i,0].parent_mode.name if hasattr(activities_plan[i, 0], 'parent_mode') else 'No Mode'}) - Duration: {activities_plan[i, 1]} minutes")
            time_counter += int(activities_plan[i, 1])  # Increment time counter by the duration of the activity in minutes

    #Now we have an outdated event np.int array; we're going to use the events/tassl/etc themselves and see how that goes
    
    # def calculate_cost(self, activities_plan_nan, remaining_time:int = 1440, v:bool=True):
    #     if not self.activities_list_set:
    #         print("Error: activities_list must be set before calculating cost")
    #         raise ValueError("AttributeError: activities_list must be set before calculating cost")
        
    #     if not self.events_row_list_set:
    #         print("Error: events_row_list must be set before calculating cost")
    #         raise ValueError("AttributeError: events_row_list must be set before calculating cost")

    #     if remaining_time not in range(0, 1441):
    #         print("Error: remaining_time must be an int between 0 and 1440")
    #         raise ValueError("AttributeError: remaining_time must be an int between 0 and 1440")

    #     cost = 0 #Logic function to calculate cost of the activities plan, count up by violations

    #     activities_plan = np.nan_to_num(activities_plan_nan, nan=0.0)  # Replace NaN values with 0.0
        
    #     if v: print("Confirming the activities plan sum is appropriate:")
        
    #     total_time_cost = (np.sum(activities_plan[:, 1]) - remaining_time) ** 2  # Sum of all activity durations plus remaining time
    #     if v: print(f"Total time cost: {total_time_cost}")
    #     cost += 10**6 * total_time_cost  # Major violation; needs to be heavily penalized

    #     #Check that each event in the day is accounted for in the activities plan
    #     for event in self.events_row_list:
            
            
            
    #         if v: print(f"For event {self.activities_list[event[0]].name}, checking if it is accounted for in the activities plan")
            
    #         closest_norm_ind = np.argmin(np.linalg.norm(activities_plan[:, :-1] - event, axis=1))  # Find the closest event in the activities plan
    #         closest_norm = np.linalg.norm(activities_plan[closest_norm_ind, :-1] - event)  # Calculate the norm of the difference

    #         cost += 10**5 * closest_norm ** 2  # Major violation; needs to be heavily penalized
    #         if v: print(f"Closest norm is {closest_norm}")

            
            
            
    #         locations_list = self.activities_list[event[0]].location_set
            
    #         location_result = [activities_plan[closest_norm_ind, 2] - locations_list[i] for i in range(len(locations_list))]  # Distance to possible locations for event
    #         location_result = np.array(location_result).T #Turns into a row vector
    #         closest_location_ind = np.argmin(np.min(location_result**2))
    #         closest_location_cost = np.min(location_result**2)

    #         cost += 10**5 * closest_location_cost # Major violation; needs to be heavily penalized
    #         if v: print(f"Closest location is {inv_locations[locations_list[closest_location_ind]]} with cost {closest_location_cost}")



    #         time_leading = activities_plan[:closest_norm_ind, 1].sum()
    #         time_wanted = (self.activities_list[event[0]].time_start.hour * 60 + self.activities_list[event[0]].time_start.minute)
    #         cost += 10**5 * (time_leading - time_wanted) ** 2  # Major violation; needs to be heavily penalized
    #         if v: print(f"Time leading up to event {self.activities_list[event[0]].name} is {time_leading} minutes; the ideal is {time_wanted}.")

    #     #Next we include cost for each goal!


            
        #Now we want to

        # return cost + np.linalg.norm(activities_plan, 2, axis=1).sum()
         

    def generate_activities_1(self, week, past:object=None, v:bool=True, seed:int=0, chunk_size:int=90):

        if not self.proposed_dist_set:
            print("Error: proposed_dist must be set before listing all activities")
            raise ValueError("AttributeError: proposed_dist must be set before listing all activities")
    
        ''''Generate's the 'activities' object to start!'''

        if past == None:
            activities = np.array([[None, 1440, None]])

        else:
            time_spent = np.sum(past[:, 1])  # Sum of all activity durations in the past
            activities = past.copy()  # Copy the past activities to start with
            activities = np.vstack((activities, [None, 1440 - time_spent, None]))  # Add the remaining time row

        if v: print(f"\n\tActivities array before adding events:\n{activities}")

        if not any ([x.name == "Day Start" for x in self.events]):
            self.events.append(Event("Day Start", modes["fx"], datetime.combine(self.date, time(0, 0)), datetime.combine(self.date, time(0, 0))))  # Add a day start event if not present

        #Randomly shuffle the events row list:
        events = self.events.copy()
        np.random.seed(seed+123123+self.date.toordinal())  # Set the random seed for reproducibility
        np.random.shuffle(events)  # Shuffle the events to randomize their order

        if v:
            print(f"\n\tOrder of events to insert:")
            for event in events:
                print(f"Event: {event.name}")

        for i, event in enumerate(events):
            time_start_minute = event.time_start.hour * 60 + event.time_start.minute
            time_est = event.time_end.hour * 60 + event.time_end.minute - time_start_minute

            if activities.shape[0] == 1:
                activities[0,:] = [None, time_start_minute, None]
                activities = np.vstack((activities, [event, time_est, None]))  # Add the event row
                activities = np.vstack((activities, [None, 1440 - time_start_minute - time_est, None]))  # Add the remaining time row

            else:
                iterate_rows = 0

                while iterate_rows < activities.shape[0] and np.sum(activities[:iterate_rows+1, 1]) < time_start_minute:  #Iterates until that row is the one whose sum is less that time_start_minutes
                    iterate_rows += 1

                if activities[iterate_rows, 0] is not None:
                    raise ValueError(f"AttributeError: Cannot insert event {event.name} at time {time_start_minute} as there is already an activity at that time.")
                
                elif activities[iterate_rows, 1] < time_est:
                    raise ValueError(f"AttributeError: Cannot insert event {event.name} at time {time_start_minute} as there is not enough time before the next activity.")
                
                before_rows = activities[0:iterate_rows, :]
                after_rows = activities[iterate_rows+1:, :]
                replace_row = activities[iterate_rows, :]

                replacement = np.array([
                        [None, time_start_minute - np.sum(before_rows[:, 1]), None],
                        [event, time_est, None],
                        [None, replace_row[1] - (time_est + time_start_minute - np.sum(before_rows[:, 1])), None]
                ])
                
                activities = np.vstack((before_rows, replacement, after_rows))  # Add the event row

        if v: 
            print(f"\n\tActivities array after adding events:")
            for row in activities:
                print(f"{row[0].name if row[0] is not None else 'None'}: {row[1]} minutes")

        self.print_activities(activities)

        i = 0
        while i < activities.shape[0]:
            if activities[i, 0] is None and activities[i, 1] > chunk_size:
                
                # Split the chunk into smaller chunks
                num_chunks = int(activities[i, 1] // chunk_size)
                remaining_time = activities[i, 1] % chunk_size
                
                new_rows = []
                for j in range(num_chunks):
                    new_rows.append([None, chunk_size, None])
                
                if remaining_time > 0:
                    new_rows.append([None, remaining_time, None])
                
                # Insert the new rows into the activities array
                activities = np.insert(activities, i + 1, new_rows, axis=0)
                
                # Remove the original row
                activities = np.delete(activities, i, axis=0)
            else:
                i += 1

        if v:
            print(f"\nActivities array after splitting chunks:")
            for row in activities:
                print(f"{row[0].name if row[0] is not None else 'None'}: {row[1]} minutes")

        locked_in_dist = self.reset_lid(activities)

        if v: 
            print(f"\n\tLocked in distribution after adding events:")
            for i, mode in enumerate(modes):
                print(f"mode: {modes[mode].name}: {locked_in_dist[i]}")

        for task in self.tasks:
            locked_in_dist[[modes[mode] for mode in modes].index(task.parent_mode)] += int(self.tasks[task].total_seconds() / 60)  # Add the task to the day's locked in distribution

        if v:
            print("\n\tDay-Level (Locked In) Tasks Added:")
            for i, mode in enumerate(modes):
                print(f"mode: {modes[mode].name}: {locked_in_dist[i]}")

        tasks = self.tasks.copy()  # Copy the tasks to avoid modifying the original dictionary
        
        for task in week.tasks: #Assuming you sort week tasks outside of this function
            if week.tasks[task] > timedelta(minutes=0):
                print(f"ADDING WEEK TASK {task.name}; {week.tasks[task]}")
                tasks, locked_in_dist = self.add_task(task, tasks, week.tasks[task], locked_in_dist, v=v)  # Add the task to the day's task
        
        if v:
            print("\n\tWeek-Level Tasks Added:")
            for i, mode in enumerate(modes):
                print(f"mode: {modes[mode].name}: {locked_in_dist[i]}")

        for mode in modes:  #Assuming you sort mode tasks outside of this function

            for task in modes[mode].tasks:
                time_remaining = self.proposed_dist[mode_ids.index(mode)] - locked_in_dist[mode_ids.index(mode)]  # Calculate the time available for this mode
                
                if time_remaining <= 0:
                    break

                if task not in self.tasks and modes[mode].tasks[task] > timedelta(minutes=0):
                    tasks, locked_in_dist = self.add_task(task, tasks, modes[mode].tasks[task], locked_in_dist, v=v)  # Add the task to the day's tasks

        if v:
            print("\n\tGlobal-Level Tasks Added:")
            for i, mode in enumerate(modes):
                print(f"mode: {modes[mode].name}: {locked_in_dist[i]}")
                print("\n")

        if v:
            print(f"\n\tProposed-Locked Difference:")
            for i, mode in enumerate(modes):
                print(f"mode: {modes[mode].name}: {self.proposed_dist[i] - locked_in_dist[i]}")
                print("\n")

        tasks_list = list(tasks.keys()).copy()  # Copy the tasks to avoid modifying the original dictionary
        np.random.shuffle(tasks_list)
        tasks = {task: tasks[task] for task in tasks_list}  # Shuffle the tasks to randomize their order

        if v:
            print(f"\n\t Tasks to be added:")
            for task in tasks:
                print(f"{task.name}")
            print("\n")

        task_indexer = 0
        time_history = 1440
        activities_history = activities.copy()

        while task_indexer < len(tasks.keys()):
            task = list(tasks.keys())[task_indexer]
            if v: print(f"Distributing task: {task.name}:\n")

            if task.name == "Healthy Sleep":
                return activities, tasks, task  # Skip meal prep as it is handled separately
            
            if (hasattr(task, 'need_before') and task.need_before is not None) or (hasattr(task, 'need_after') and task.need_after is not None):
                parent_task = get_parent_task(activities, tasks, task)
                if parent_task not in activities[:, 0]:
                    tasks[task] = (tasks.pop(task))  # Move the current task to the end of the list to try again later
                    task_indexer -= 1  
                else:
                    while tasks[task].total_seconds() / 60 > 0:
                        activities = insert_with_parent(activities, tasks, task)
                        activities = clear_zeros(activities)  # Clear any rows with zero duration

            elif hasattr(task, "ideal_time"):

                while tasks[task].total_seconds() / 60 > 0:

                    if not any([activities[i, 0] is task for i in range(activities.shape[0])]):
                        differences_eligible, direction = get_differences_eligible(activities, task)
                        insert_index = np.argmin(differences_eligible)
                        if v: print(f"Insert index: {insert_index}, with value {differences_eligible[insert_index]}")

                        activities = insert_at_none(activities, tasks, task, insert_index, direction)

                    else:
                        
                        activities = insert_orphan(activities, tasks, task)
                    
                    activities = clear_zeros(activities)  # Clear any rows with zero duration
                    
            else:   #No ideal time attribute
                if v: print(f"Task {task.name} does not have an ideal_time attribute; checking remaining tasks for ideal_time attribute.")

                remaining_tasks = list(tasks.keys())[task_indexer+1:]
                if any(hasattr(t, "ideal_time") for t in remaining_tasks):
                    if v: print(f"Found tasks with ideal_time attribute in the remaining tasks, continuing to distribute them.")
                    
                    tasks[task] = (tasks.pop(task)) #Move the current task to the end of the list to try again later
                    task_indexer -= 1

                else:
                    if task.name == "Test-Catchall for Flex" or task.name == "Test-Catchall for Travel":
                        direction = 'right'
                    else:
                        direction = 'left'
                        
                    while tasks[task].total_seconds() / 60 > 0:

                        activities = insert_anywhere(activities, tasks, task, direction=direction)

                        event = self.events[0]
            
            event = self.events[0] if len(self.events) > 0 else None
            event_index = activities[:, 0].tolist().index(event)
            
            if np.sum(activities[:,1]) != time_history:
                print(f"{task.name}\n")
                self.print_activities(activities_history)
                print("----------------------------")
                self.print_activities(activities)
                raise ValueError(f"Activities do not sum up to {time_history} minutes after distributing task {task.name}. Please check your distribution logic.")

            elif not self.check_events(activities):
                print(f"{task.name} failed")
                return activities, tasks
            
            else:
                print(f"{task.name} passed")
            
            activities_history = activities.copy()  # Save the current state of activities for the next iteration
            
            task_indexer += 1



        #We need to iterate through and add travel time!

        
        if v: print(f"Minutes sum: {np.sum(activities[:, 1])} minutes")

        return clear_zeros(activities), tasks, task

    def add_task(self, task, tasks, time_est, locked_in_dist, v=True):
        if not self.proposed_dist_set:
            print("Error: proposed_dist must be set before adding tasks")
            raise ValueError("AttributeError: proposed_dist must be set before adding tasks")

        mode_index = mode_names.index(task.parent_mode.name)  # Get the index of the mode in the mode_names list
        time_to_distribute = self.proposed_dist[mode_index] - locked_in_dist[mode_index]  # Calculate the time available for this mode

        if time_to_distribute <= 0:
            return tasks, locked_in_dist
        
        elif time_to_distribute < int(time_est.total_seconds() / 60):
            tasks[task] = timedelta(minutes=int(time_to_distribute))  # Add the task with the available time
            locked_in_dist[mode_index] += time_to_distribute  # Update the locked-in distribution

        else:
            tasks[task] = time_est  # Add the task with its full time estimate
            locked_in_dist[mode_index] += int(time_est.total_seconds() / 60)  # Update the locked-in distribution

        return tasks, locked_in_dist
        
    
    def check_events(self, activities):
        for event in self.events:
            check_index = [i for i, x in enumerate(activities[:, 0]) if x is event]
            if len(check_index) != 1:
                return False
            event_time_should = event.time_start.hour * 60 + event.time_start.minute
            event_time_planned = np.sum(activities[:check_index[0], 1])  # Sum of all activity durations before the event
            if event_time_should != event_time_planned:
                return False
        return True


    #We'll re-implement this if we decide that vectors in this format are the way to go.
   
    # def generate_events_vector(self):
    #     if not self.activities_list_set:
    #         print("Error: activities_list must be set before generating events vector")
    #         raise ValueError("AttributeError: activities_list must be set before generating events vector")
        
    #     vector_list = []

    #     for i, activity in enumerate(self.activities_list):
    #         if isinstance(activity, Event):
    #             event_vector = np.full(24*60, np.nan, dtype=int)  # 24 hours * 60 minutes
    #             start_minute = (activity.time_start.time().hour * 60) + activity.time_start.time().minute
    #             end_minute = (activity.time_end.time().hour * 60) + activity.time_end.time().minute - 1 # -1 to include the end minute in the event

    #             event_vector[start_minute:end_minute] = i

    #             vector_list.append(event_vector)
        
    #     return sum(vector_list)

    # We'll re-implement this if we decide that vectors in this format are the way to go.
    
    # def print_vector(self, plan_vector):
    #     if not self.activities_list_set:
    #         print("Error: activities_list must be set before printing vector")
    #         raise ValueError("AttributeError: activities_list must be set before printing vector")
        
    #     for i in range(len(plan_vector)):
    #         if i == 0:
    #             print(f"{time(hour=i//60, minute=i%60)}:")

    #             if plan_vector[i] in range(len(self.activities_list)):
    #                 print(f" {self.activities_list[plan_vector[i]].name} ({self.activities_list[plan_vector[i]].parent_mode.name})")

    #             else:
    #                 print(" No activity planned")

    #         elif plan_vector[i] != plan_vector[i-1]:
    #             print(f"{time(hour=i//60, minute=i%60)}: ")

    #             if plan_vector[i] in range(len(self.activities_list)):
    #                 print(f" {self.activities_list[plan_vector[i]].name} ({self.activities_list[plan_vector[i]].parent_mode.name})")

    #             else:
    #                 print(" No activity planned")
    
    def reset_lid(self, activities):
        if not isinstance(activities, np.ndarray):
            raise TypeError("AttributeError: activities must be a numpy array")
        
        locked_in_dist = np.zeros(len(modes), dtype=int)  # Reset the locked-in distribution to zero for each mode
        for i, activity in enumerate(activities[:,0]):
            if activity is not None:
                index = [value for value in modes.values()].index(activity.parent_mode)  # Get the index of the mode in the modes list
                locked_in_dist[index] += int(activities[i, 1])  # Add the activity's duration to the locked-in distribution for the corresponding mode

        return locked_in_dist
            
                
                
            



class Present_day:
    def __init__(self, day_future:Day):
        self.date = day_future.date

class Past_day:
    def __init__(self, day_present:Present_day):
        self.date = day_present.date




In [10]:
#Define Week Class and intializes our guinea pig week

class Week:

    def __init__(self, start_date:date, name:str, daily_flex:timedelta = timedelta(hours=1.5), daily_travel:timedelta = timedelta(hours=1), modes_min:timedelta = timedelta(hours=1)):
        if not isinstance(start_date, date):
            raise TypeError("AttributeError: start_date and end_date must be date objects")
        
        #Make sure that start date is a Sunday
        if start_date.weekday() != 6:
            raise ValueError("AttributeError: start_date must be a Sunday")
        
        self.modes_min = modes_min
        
        self.name = name
        
        self.start_date = start_date
        self.end_date = start_date + timedelta(days=6)

        self.days = {
            "Sun" : Day(self.get_date("Sun")),
            "Mon" : Day(self.get_date("Mon")),
            "Tue" : Day(self.get_date("Tue")),
            "Wed" : Day(self.get_date("Wed")),
            "Thu" : Day(self.get_date("Thu")),
            "Fri" : Day(self.get_date("Fri")),
            "Sat" : Day(self.get_date("Sat"))
        }

        #Dict's keyed by time mode
        self.ideal = {}
        self.weights = {}
        
        #Timedelta
        self.daily_flex = daily_flex
        self.daily_travel = daily_travel

        #List of Recurring Events
        self.event_strs = []
        self.events_list_recurring = []
        
        #List of one-time events identified as happening this week
        self.events_list_unique = []

        #IMPORTANT! Tasks that are weekly
        self.tasks = {}
        for mode in modes:
            for task in modes[mode].weekly_tasks:
                self.tasks[task] = task.time_est

            for task in modes[mode].tasks:
                if task.deadline is not None and task.deadline.date() >= self.start_date and task.deadline.date() <= self.end_date + timedelta(weeks=1):
                    if task not in self.tasks:
                        self.tasks[task] = modes[mode].tasks[task]

        self.locked_in_dist = np.zeros(len(modes), dtype=int)
        self.locked_in_task_dist = np.zeros(len(modes), dtype=int)
        self.ideal_dist = np.zeros(len(modes), dtype=int)
        self.remaining_to_distribute = np.zeros(len(modes), dtype=int)
        self.ideal_deficit = np.zeros(len(modes), dtype=int)
        self.minimum_deficit = np.zeros(len(modes), dtype=int)

        self.remaining_to_distribute_days = {
            day : np.zeros(len(modes), dtype=int) for day in self.days
        }

        self.events_generated = False
        self.ideal_set = False
        self.weights_set = False
        self.recurring_events_distributed = False
        self.unique_events_distributed = False
        self.locked_in_days = False
        self.locked_in_tasks = False
        self.locked_in_dist_ok = False
        self.ideal_dist_distributed = False
        self.ideal_dist_distributed_to_days = False
        self.ideal_dist_ok = False

    def get_date(self, day:str):
        days_dict = {
            "Sun" : 6,
            "Mon" : 0,
            "Tue" : 1,
            "Wed" : 2,
            "Thu" : 3,
            "Fri" : 4,
            "Sat" : 5
        }

        if day not in days_dict:
            raise ValueError("AttributeError: day must be one of the following: Sun, Mon, Tue, Wed, Thu, Fri, Sat")
        day_index = days_dict[day]

        for i in range(7):
            if (self.start_date + timedelta(days=i)).weekday() == day_index:
                return self.start_date + timedelta(days=i)
            
    def generate_events(self):
        if self.events_generated:
            print("Events already generated for this week")
            return
        
        for event_str in self.event_strs:

            name = event_str[0] + f" (Week: {self.start_date.strftime('%Y-%m-%d')})"
            
            if any(x.name == name for x in event_str[1].events):
                for x in event_str[1].events:
                    if x.name == name:
                        print("Found Duplicate Event:")
                        x.print_event_info()
                        self.events_list_recurring.append(x)
                        print(f"Event assymilated into week: {x in self.events_list_recurring}")
                        continue
            else:
                event = Event(event_str[0] + f" (Week: {self.start_date.strftime('%Y-%m-%d')})", event_str[1], datetime.combine(self.get_date(event_str[2]), event_str[3]), datetime.combine(self.get_date(event_str[2]), event_str[4]), location_set=event_str[5])
                #Reject the event if a duplicate was found, i.e. if parent_mode is not an attribute:
                if not hasattr(event, 'parent_mode'):
                    print(f"Event {event_str[0]} already exists in {event_str[1].name} mode")
                    continue
                if event is not None: self.events_list_recurring.append(event)

        self.events_generated = True

    def clear_events(self):
        if not self.events_generated:
            print("No events to clear for this week")
            return
        
        for event in self.events_list_recurring:
            event.parent_mode.events.remove(event)
            del event

        self.events_list_recurring.clear()
        self.events_generated = False
    
    def hunt_unique_events(self):
        if len(self.days) == 0:
            print("It's recommended to generate days before hunting unique events")
            raise ValueError("AttributeError: days must be a list of Day objects")
        
        for mode in modes:
            for event in modes[mode].events:
                if any([event.time_start.date() == self.days[day].date for day in self.days]) and event not in self.events_list_recurring:
                    if event not in self.events_list_unique: self.events_list_unique.append(event)
                    if event not in self.days[get_weekday_str(event.time_start.date())].events: self.days[get_weekday_str(event.time_start.date())].events.append(event)

        self.unique_events_distributed = True

    def distribute_recurring_events(self):
        if not self.events_generated:
            raise ValueError("AttributeError: events must be generated before distributing recurring events")
        
        if len(self.days) == 0:
            print("It's recommended to generate days before distributing recurring events")
            raise ValueError("AttributeError: days must be a list of Day objects")
        
        for event in self.events_list_recurring:
            if event not in self.days[get_weekday_str(event.time_start.date())].events:
                self.days[get_weekday_str(event.time_start.date())].events.append(event)
                print(f"Distributed {event.name} to {get_weekday_str(event.time_start.date())} day")
            else:
                print(f"{event.name} already exists in {get_weekday_str(event.time_start.date())} day")

        self.recurring_events_distributed = True

    def set_ideal(self, ideals:dict):
        if self.ideal_set:
            print("Ideals already set for this week, skipping")
            return
        
        if not isinstance(ideals, dict):
            raise TypeError("AttributeError: ideals must be a dict")
        
        self.ideal = ideals

        self.ideal_set = True

    def set_weights(self, weights:dict):
        if self.weights_set:
            print("Weights already set for this week, skipping")
            return
        
        if not isinstance(weights, dict):
            raise TypeError("AttributeError: weights must be a dict")
        
        self.weights = weights

        self.weights_set = True

    def get_days_remaining(self):
        #Count the number of days in days dict that are of type Day
        return [day for day in self.days if isinstance(self.days[day], Day)]
    
    def audit_events_distributed(self):
        if self.unique_events_distributed and self.recurring_events_distributed:
            
            for day in self.days:
                if isinstance(self.days[day], Day):
                    self.days[day].events_set = True

        else: raise AttributeError("AttributeError: unique_events_distributed and recurring_events_distributed must be True before auditing events distributed")

    def set_locked_in_dist_days(self):
        if not self.unique_events_distributed or not self.recurring_events_distributed:
            print("Please distribute unique and recurring events before setting locked in dist for days")
            raise ValueError("AttributeError: unique_events_distributed and recurring_events_distributed must be True before setting locked in dist for days")

        for day in self.days:
            if isinstance(self.days[day], Day):
                
                if self.days[day].locked_in_dist_set:
                    print(f"Locked in dist already set for {day}")
                    continue
                
                self.days[day].set_locked_in_dist()

    def set_locked_in_dist(self):
        if not self.unique_events_distributed or not self.recurring_events_distributed:
            print("Please distribute unique and recurring events before setting locked in dist for days")
            raise ValueError("AttributeError: unique_events_distributed and recurring_events_distributed must be True before setting locked in dist for days")


        if not self.locked_in_days:   
            for day in self.days:
                if isinstance(self.days[day], Day):
                    if not self.days[day].locked_in_dist_set:
                        print(f"Please set locked in dist for {day} before setting locked in dist for week")
                        return
                    
                    if not self.days[day].locked_in_ok:
                        print(f"Locked in dist for {day} is not ok, please check your daily ideals, events, and tasks")
                        return
                    
                    #If we get here, the day is ok to be counted towards the week's locked in dist
                    if not hasattr(self, 'locked_in_dist'):
                        raise ValueError("AttributeError: week must have a locked_in_dist attribute")
                    
                    self.locked_in_dist += self.days[day].locked_in_dist

                elif isinstance(self.days[day], Present_day):
                    print(f"Present day {day} does not have a locked in dist, skipping")
                    continue

                elif isinstance(self.days[day], Past_day):
                    print(f"Interpreting locked in dist for past day as good, provided we can construct object with no faults from the onset")
                    self.locked_in_dist += self.days[day].locked_in_dist

            self.locked_in_days = True

        else:
            print("Locked in dist already set for week, skipping")
            return

        #Need a dict mode to collect form tasks
        dist_dict = {modes[mode]: int(0) for mode in modes}
        
        if not self.locked_in_tasks:
            for task in self.tasks:
                dist_dict[task.parent_mode] += int(self.tasks[task].total_seconds() / 60)
        else:
            print("Locked in tasks already set, skipping")
            return

        self.locked_in_task_dist += np.array([dist_dict[modes[mode]] for mode in modes], dtype=int)

        self.locked_in_tasks = True

    def audit_locked_in_dist(self):

        if not self.locked_in_days:
            print("Please set locked in dist before auditing locked in dist")
            raise ValueError("AttributeError: locked_in_dist must be set before auditing locked in dist")
        
        if not self.locked_in_dist_ok:
            total_minutes_locked_in = np.sum(self.locked_in_dist)

            print(f"Total minutes locked in: {total_minutes_locked_in}")

            total_minutes_locked_in += np.sum(self.locked_in_task_dist)

            days_remaining = len(self.get_days_remaining())

            if total_minutes_locked_in > (days_remaining * 24 * 60):
                print(f"Warning: Total minutes locked in ({total_minutes_locked_in}) exceeds {days_remaining} days. Please check your daily ideals, events, and tasks.")
                self.locked_in_dist_ok = False
            else:
                print(f"Total minutes locked in ({total_minutes_locked_in}) is within {days_remaining * 60 * 24} minutes. Good to go!")
                self.locked_in_dist_ok = True

        else:
            print("Locked in dist already audited, skipping")
            return
                
    def distribute_ideals_dist(self):
        if self.ideal_dist_distributed:
            print("Ideals dist already distributed, skipping")
            return
        
        if not self.locked_in_dist_ok:
            print("Please audit locked in dist before distributing ideals dist")
            return
        
        if not self.locked_in_tasks:
            print("Please set locked in tasks before distributing ideals dist")
            return

        if not self.weights_set:
            print("Please set weights before distributing ideals dist")
            return

        days_remaining = len(self.get_days_remaining())

        if days_remaining == 0:
            print("No days remaining to distribute ideals dist, skipping")
            return
        
        weights_np = np.array([self.weights[mode] for mode in modes], dtype=float)
        mins_np = np.array([int(self.modes_min.total_seconds() / 60) for mode in modes], dtype=int)
        mins_np[-3] = int((self.daily_flex * days_remaining).total_seconds() / 60)
        mins_np[-2] = int((self.daily_travel * days_remaining).total_seconds() / 60)

        print(f"Making sure to distribute flex time, {mins_np[-3]} minutes accross {days_remaining} days, or {self.daily_flex} per day.")

        mins_np[-1] = 0 # Last mode is 'test', which we don't want to distribute ideals for
        ideal_np = np.array([int(self.ideal[mode].total_seconds() / 60) for mode in modes], dtype=int)
        
        x = cp.Variable(len(modes), integer=True)   #Total not - locked - in time for each mode in minutes
        s = cp.Variable(len(modes), nonneg=True)  #Slack variables for each mode in minutes; HEVAILY penalized if not 0
        u = cp.Variable(len(modes), nonneg=True)

        constraints = [cp.sum(x) + np.sum(self.locked_in_dist) + np.sum(self.locked_in_task_dist) == days_remaining * 24 * 60]
        constraints.append(x >= 0)
        constraints.append(x + self.locked_in_dist + self.locked_in_task_dist + s >= mins_np)
        constraints.append(u >= ideal_np - (x + self.locked_in_dist + self.locked_in_task_dist))

        objective = cp.Minimize(cp.sum(cp.multiply(weights_np, cp.square(u))+1000*cp.sum(cp.square(s))))

        problem = cp.Problem(objective, constraints)
        problem.solve(verbose=True, solver=cp.GUROBI)
        if problem.status != cp.OPTIMAL:
            print("Problem not solved to optimality, please check your constraints and objective function")
            return
        
        else:
            print("Problem solved to optimality, distributing ideals dist")
        
        self.remaining_to_distribute = x.value
        self.ideal_deficit = u.value
        self.minimum_deficit = s.value

        self.ideal_dist_distributed = True
        

    def print_distribution_plan(self):
        if not self.ideal_dist_distributed:
            print("Please distribute ideals dist before printing distribution plan")
            return
        
        print(f"Distribution Plan for Week: {self.name} ({self.start_date.strftime('%Y-%m-%d')} to {self.end_date.strftime('%Y-%m-%d')})")
        for i, mode in enumerate(modes):
            print(f"{modes[mode].name}:")
            print(f"  Locked In: {self.locked_in_dist[i] + self.locked_in_task_dist[i]} minutes")
            print(f"  Remaining to Distribute: {self.remaining_to_distribute[i]} minutes")
            print(f"  Ideal Deficit: {self.ideal_deficit[i]} minutes")
            print(f"  Minimum Deficit: {self.minimum_deficit[i]} minutes")

    def audit_ideals_dist(self):
        if self.ideal_dist_ok:
            print(f"Already audited the ideal distribution. No need. Skipping.")
            return
        
        if not self.ideal_dist_distributed:
            print("Please distribute ideals dist before auditing ideals dist")
            return
        
        #Print a header so that this part is going to be easy to find in a huge scrollable element
        print("\n".join(["#" * 60, "#{:^58}#".format("✨ WELCOME TO THE GUROBI ZONE ✨"), "#" * 60]))

        #Then we make sure that all the distributions add up.

        days_remaining = len(self.get_days_remaining())
        print(f"At {days_remaining} days, we're looking at distributing {days_remaining * 24 * 60} minutes of time across the modes.")

        total_minutes_locked_in = np.sum(self.locked_in_dist) + np.sum(self.locked_in_task_dist)
        print(f"Total minutes locked in: {total_minutes_locked_in}")

        total_minutes_remaining = np.sum(self.remaining_to_distribute)
        print(f"Total minutes remaining to distribute: {total_minutes_remaining}")

        print(f"Now the buck stops here: {total_minutes_locked_in + total_minutes_remaining} minutes should equal {days_remaining * 24 * 60} minutes.")
        print(f"{total_minutes_locked_in + total_minutes_remaining == days_remaining * 24 * 60}!! {"Let's Go!" if total_minutes_locked_in + total_minutes_remaining == days_remaining * 24 * 60 else "oops."}")

        if total_minutes_locked_in + total_minutes_remaining != days_remaining * 24 * 60:
            self.ideal_dist_ok = False

        else:
            self.ideal_dist_ok = True
        
    def distribute_ideal_dist_to_days(self):
        
        days_revert = self.days.copy()  # Save the original days dict to revert back if needed
        
        
        if not self.ideal_dist_distributed:
            print("Please distribute ideals dist before distributing to days")
            return
        
        if self.ideal_dist_distributed_to_days:
            print("Ideals dist already distributed to days, skipping")
            return
        
        #This is the obejct from which we will be distributing the ideal distribution to the days.
        minutes_to_distribute = np.copy(self.remaining_to_distribute + self.locked_in_task_dist)

        print(f"Minutes to distribute before distributing to days: {minutes_to_distribute}")

        for day in self.days:
            if isinstance(self.days[day], Day):
                if not self.days[day].locked_in_dist_set:
                    print(f"Please set locked in dist for {day} before distributing ideals dist to days")
                    self.days = days_revert  # Revert to the original days dict
                    return
                
                if not self.days[day].locked_in_ok:
                    print(f"Locked in dist for {day} is not ok, please check your daily ideals, events, and tasks")
                    self.days = days_revert
                    return
                
                #Distribute the ideal distribution to the day
                self.days[day].set_proposed_dist_lean(minutes_to_distribute)

        print(f"Minutes to distribute after distributing to days (lean): {minutes_to_distribute}")

        for day in self.days:
            if isinstance(self.days[day], Day):
                self.days[day].set_proposed_dist_greedy(minutes_to_distribute)

        print(f"Minutes to distribute after distributing to days (greedy): {minutes_to_distribute}")

        self.ideal_dist_distributed_to_days = True

    def distribute_tasks(self):
        if not self.ideal_dist_distributed_to_days:
            print("Please distribute ideals dist to days before distributing tasks")
            raise ValueError("AttributeError: ideal_dist_distributed_to_days must be True before distributing tasks")
        
        
        


        

        

    


        

In [11]:
#If there's no week called week1, then we create it.
if 'week1' in globals():
    del week1

week1 = Week(date(2025, 6, 8), "Guinea Pig Week")

week1.set_ideal({
    "rd" : timedelta(hours=7),  #read
    "vs" : timedelta(hours=4),  #videogames & shows
    "hb" : timedelta(hours=10),  #hobbies
    "mh" : timedelta(hours=2),  #mental health
    "ph" : timedelta(hours=2),  #physical health
    "hg" : timedelta(hours=2),  #hygiene
    "ml" : timedelta(hours=5),  #meals
    "fn" : timedelta(hours=6),  #friends
    "st" : timedelta(hours=5),  #spiritual
    "hm" : timedelta(hours=5),  #home
    "ms" : timedelta(hours=5),  #music
    "fi" : timedelta(hours=2),  #finance
    "wk" : timedelta(hours=50),  #work
    "sl" : timedelta(hours=56),  #sleep
    "pl" : timedelta(hours=7),  #planning
    "sp" : timedelta(hours=3),  #sports
    "fm" : timedelta(hours=3),  #family
    "fx" : timedelta(hours=0),  #flex
    "tv" : timedelta(hours=0),  #travel
    "test" : timedelta(hours=0)  #test
})

week1.set_weights({
    "rd" : 0.8,  #read
    "vs" : 0.3,  #videogames & shows
    "hb" : 0.8,  #hobbies
    "mh" : 0.6,  #mental health
    "ph" : 0.6,  #physical health
    "hg" : 0.6,  #hygiene
    "ml" : 0.6,  #meals
    "fn" : 0.6,  #friends
    "st" : 0.9,  #spiritual
    "hm" : 0.6,  #home
    "ms" : 0.6,  #music
    "fi" : 0.6,  #finance
    "wk" : 0.9,  #work
    "sl" : 1,  #sleep
    "pl" : 0.6,  #planning
    "sp" : 0.3,  #sports
    "fm" : 0.5,  #family
    "fx" : 0,  #flex
    "tv" : 0,  #travel
    "test" : 0  #test
})



week_day_ideals = {
    "rd" : timedelta(hours=0),  #read
    "vs" : timedelta(hours=0),  #videogames & shows
    "hb" : timedelta(hours=0),  #hobbies
    "mh" : timedelta(hours=0),  #mental health
    "ph" : timedelta(hours=0),  #physical health
    "hg" : timedelta(hours=0),  #hygiene
    "ml" : timedelta(hours=0),  #meals
    "fn" : timedelta(hours=0),  #friends
    "st" : timedelta(hours=0),  #spiritual
    "hm" : timedelta(hours=0),  #home
    "ms" : timedelta(hours=0),  #music
    "fi" : timedelta(hours=0),  #finance
    "wk" : timedelta(hours=8),  #work
    "sl" : timedelta(hours=0),  #sleep
    "pl" : timedelta(hours=0),  #planning
    "sp" : timedelta(hours=0),  #sports
    "fm" : timedelta(hours=0),  #family
    #For flex and travel, originally it was thought that they should be defined in the set_ideals helper function. But to keep the order of the dict consistent, we set them here.
    "fx" : timedelta(hours=1),  #flex
    "tv" : timedelta(hours=1),  #travel #Then if they need changing, it's easy to change them in the set_ideals function.
    "test" : timedelta(hours=0)  #test
}

#Saturday's one emphasized time mode is hobbies, so we set that here.
saturday_ideals = {
    "rd" : timedelta(hours=0),  #read
    "vs" : timedelta(hours=0),  #videogames & shows
    "hb" : timedelta(hours=5),  #hobbies
    "mh" : timedelta(hours=0),  #mental health
    "ph" : timedelta(hours=0),  #physical health
    "hg" : timedelta(hours=0),  #hygiene
    "ml" : timedelta(hours=0),  #meals
    "fn" : timedelta(hours=0),  #friends
    "st" : timedelta(hours=0),  #spiritual
    "hm" : timedelta(hours=0),  #home
    "ms" : timedelta(hours=0),  #music
    "fi" : timedelta(hours=0),  #finance
    "wk" : timedelta(hours=0),  #work
    "sl" : timedelta(hours=0),  #sleep
    "pl" : timedelta(hours=0),  #planning
    "sp" : timedelta(hours=0),  #sports
    "fm" : timedelta(hours=0),  #family
    "fx" : timedelta(hours=1),  #flex
    "tv" : timedelta(hours=1),  #travel
    "test" : timedelta(hours=0)  #test
}

#And Sunday'set one emphasized time mode is spiritual, so we set that here. 
sunday_ideals = {
    "rd" : timedelta(hours=0),  #read
    "vs" : timedelta(hours=0),  #videogames & shows
    "hb" : timedelta(hours=0),  #hobbies
    "mh" : timedelta(hours=0),  #mental health
    "ph" : timedelta(hours=0),  #physical health
    "hg" : timedelta(hours=0),  #hygiene
    "ml" : timedelta(hours=0),  #meals
    "fn" : timedelta(hours=0),  #friends
    "st" : timedelta(hours=2),  #spiritual
    "hm" : timedelta(hours=0),  #home
    "ms" : timedelta(hours=0),  #music
    "fi" : timedelta(hours=0),  #finance
    "wk" : timedelta(hours=0),  #work
    "sl" : timedelta(hours=0),  #sleep
    "pl" : timedelta(hours=0),  #planning
    "sp" : timedelta(hours=0),  #sports
    "fm" : timedelta(hours=0),  #family
    "fx" : timedelta(hours=1),  #flex
    "tv" : timedelta(hours=1),  #travel
    "test" : timedelta(hours=0)  #test
}

#Great! Now we throw them into a list:
daily_ideals = [
    
    sunday_ideals,      #Sunday
    week_day_ideals,  #Monday
    week_day_ideals,  #Tuesday
    week_day_ideals,  #Wednesday
    week_day_ideals,  #Thursday
    week_day_ideals,  #Friday
    saturday_ideals   #Saturday
    
]

for i, x in enumerate(week1.days):
    week1.days[x].set_ideals(daily_ideals[i])

#Event("Denver Wellness Appointment", modes["mh"], datetime(2025, 6, 11, 14, 30), datetime(2025, 6, 11, 15, 15), location_set=["Interlocken"])

# Weekly_event("Mass", modes['st'], type1, time(hour=10), time(hour=11), 6, ["Boulder Hill"])
# Weekly_event("Lecture", modes['wk'], type1, time(hour=12), time(hour=13), 0, ["CU Engineering"])
# Weekly_event("Work Meeting", modes['wk'], type1, time(hour=14), time(hour=15), 0, ["CU Engineering"])

week1.event_strs = [
    ["Mass", modes['st'], "Sun", time(hour=10), time(hour=11), ["Boulder Hill"]],
    ["Lecture 1", modes['wk'], "Mon", time(hour=12), time(hour=13), ["CU Engineering"]],
    ["Work Meeting 1", modes['wk'], "Tue", time(hour=14), time(hour=15), ["CU Engineering"]],
    ["Lecture 2", modes['wk'], "Wed", time(hour=14), time(hour=15), ["CU Engineering"]],
    ["Work Meeting 2", modes['wk'], "Thu", time(hour=14), time(hour=15), ["CU Duane"]],
    ["Lecture 3", modes['wk'], "Fri", time(hour=14), time(hour=15), ["CU Math"]],
    ["Work Meeting 3", modes['wk'], "Sat", time(hour=14), time(hour=15), ["CU IMIG"]],
    ["Club Meeting", modes['hb'], "Sun", time(hour=16), time(hour=17), ["Union Station"]],
]

week1.generate_events()
week1.hunt_unique_events()
week1.distribute_recurring_events()
week1.audit_events_distributed()
week1.set_locked_in_dist_days()
week1.set_locked_in_dist()
week1.audit_locked_in_dist()
week1.distribute_ideals_dist()
week1.print_distribution_plan()
week1.audit_ideals_dist()
week1.distribute_ideal_dist_to_days()


Creating new day object
Creating new day object
Creating new day object
Creating new day object
Creating new day object
Creating new day object
Creating new day object
Distributed Mass (Week: 2025-06-08) to Sun day
Distributed Lecture 1 (Week: 2025-06-08) to Mon day
Distributed Work Meeting 1 (Week: 2025-06-08) to Tue day
Distributed Lecture 2 (Week: 2025-06-08) to Wed day
Distributed Work Meeting 2 (Week: 2025-06-08) to Thu day
Distributed Lecture 3 (Week: 2025-06-08) to Fri day
Distributed Work Meeting 3 (Week: 2025-06-08) to Sat day
Distributed Club Meeting (Week: 2025-06-08) to Sun day
Total minutes locked in: 5985
Total minutes locked in (6165) is within 10080 minutes. Good to go!
Making sure to distribute flex time, 630 minutes accross 7 days, or 1:30:00 per day.
                                     CVXPY                                     
                                     v1.5.3                                    
(CVXPY) Jul 05 06:48:31 AM: Your problem has 60 variables, 6

In [12]:
self = week1.days["Sun"]

activities, tasks, task = week1.days["Sun"].generate_activities_1(week1)

self.print_activities(activities)


	Activities array before adding events:
[[None 1440 None]]

	Order of events to insert:
Event: Mass (Week: 2025-06-08)
Event: Day Start
Event: Club Meeting (Week: 2025-06-08)

	Activities array after adding events:
None: 0 minutes
Day Start: 0 minutes
None: 600 minutes
Mass (Week: 2025-06-08): 60 minutes
None: 300 minutes
Club Meeting (Week: 2025-06-08): 60 minutes
None: 420 minutes
00:00:00: None (No Mode) - Duration: 0 minutes
00:00:00: Day Start (flex) - Duration: 0 minutes
00:00:00: None (No Mode) - Duration: 600 minutes
10:00:00: Mass (Week: 2025-06-08) (spiritual) - Duration: 60 minutes
11:00:00: None (No Mode) - Duration: 300 minutes
16:00:00: Club Meeting (Week: 2025-06-08) (hobbies) - Duration: 60 minutes
17:00:00: None (No Mode) - Duration: 420 minutes

Activities array after splitting chunks:
None: 0 minutes
Day Start: 0 minutes
None: 90 minutes
None: 90 minutes
None: 90 minutes
None: 90 minutes
None: 90 minutes
None: 90 minutes
None: 60 minutes
Mass (Week: 2025-06-08): 60 

In [13]:
differences_eligible, direction = get_differences_eligible(activities, task)
insert_index = np.argmin(differences_eligible)

activities = insert_at_none(activities, tasks, task, insert_index, direction)

self.print_activities(activities)

00:00:00: None (No Mode) - Duration: 0 minutes
00:00:00: Day Start (flex) - Duration: 0 minutes
00:00:00: None (No Mode) - Duration: 90 minutes
01:30:00: None (No Mode) - Duration: 90 minutes
03:00:00: Healthy Sleep (sleep) - Duration: 90 minutes
04:30:00: None (No Mode) - Duration: 90 minutes
06:00:00: None (No Mode) - Duration: 90 minutes
07:30:00: None (No Mode) - Duration: 90 minutes
09:00:00: None (No Mode) - Duration: 60 minutes
10:00:00: Mass (Week: 2025-06-08) (spiritual) - Duration: 60 minutes
11:00:00: None (No Mode) - Duration: 90 minutes
12:30:00: None (No Mode) - Duration: 90 minutes
14:00:00: None (No Mode) - Duration: 90 minutes
15:30:00: None (No Mode) - Duration: 30 minutes
16:00:00: Club Meeting (Week: 2025-06-08) (hobbies) - Duration: 60 minutes
17:00:00: None (No Mode) - Duration: 90 minutes
18:30:00: None (No Mode) - Duration: 90 minutes
20:00:00: None (No Mode) - Duration: 90 minutes
21:30:00: None (No Mode) - Duration: 90 minutes
23:00:00: None (No Mode) - Durati

In [14]:
activities, time_inserted, movable_task  = insert_task(activities, tasks, task, is_self=True) #Insert the task where it should go
start_index = get_anchor_index(activities, task, task, is_self=True)  #Re-gather the index of the inserted task
found_index, found_direction = search_for_nearest_task_like(activities, start_index, movable_task, direction="both")    #Re-gather the movable task's index and direction
found_index_none, found_direction_none = search_for_nearest_task_like(activities, found_index, task_like=None, direction=found_direction)   #Find the nearest None task to the movable task
activities = shift_movable(activities, found_index, found_index_none, time_inserted, direction=found_direction_none)    #Move the activity that's allowed to move to the available none space
start_index = get_anchor_index(activities, task, task, is_self=True, last_direction=found_direction)  #Re-gather the start index of the task that's been inserted

print(f"Anchor index: {start_index}")
self.print_activities(activities)


# end_index = search_for_nearest_task_like(activities, start_index, task_like=movable_task, direction=found_direction)[0] #Re-gather the end of the window for activities to be shifted
# activities, peeled_queue , event_queue = peel_activities(activities, start_index, end_index, direction=found_direction, time_inserted=time_inserted)    #We peel the events into tasks and events

self.print_activities(activities)

90
Anchor index: 5
00:00:00: None (No Mode) - Duration: 0 minutes
00:00:00: Day Start (flex) - Duration: 0 minutes
00:00:00: None (No Mode) - Duration: 90 minutes
01:30:00: None (No Mode) - Duration: 0 minutes
01:30:00: Healthy Sleep (sleep) - Duration: 90 minutes
03:00:00: Healthy Sleep (sleep) - Duration: 90 minutes
04:30:00: None (No Mode) - Duration: 90 minutes
06:00:00: None (No Mode) - Duration: 90 minutes
07:30:00: None (No Mode) - Duration: 90 minutes
09:00:00: None (No Mode) - Duration: 60 minutes
10:00:00: Mass (Week: 2025-06-08) (spiritual) - Duration: 60 minutes
11:00:00: None (No Mode) - Duration: 90 minutes
12:30:00: None (No Mode) - Duration: 90 minutes
14:00:00: None (No Mode) - Duration: 90 minutes
15:30:00: None (No Mode) - Duration: 30 minutes
16:00:00: Club Meeting (Week: 2025-06-08) (hobbies) - Duration: 60 minutes
17:00:00: None (No Mode) - Duration: 90 minutes
18:30:00: None (No Mode) - Duration: 90 minutes
20:00:00: None (No Mode) - Duration: 90 minutes
21:30:00